# Sales Analysis and Prediction

In [ ]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sns
color = sns.color_palette()

from datetime import datetime
from datetime import timedelta
import math
import random
import glob # for finding pathnames matching a specified pattern
import os
import gc # garbage collection, release memory when deleting variables
import re

from scipy import stats
from scipy.stats import norm, skew #for some statistics

import copy # use df_copy = copy.deepcopy(df) to copy a dataframe (e.g. called "df"), or use: df_copy = df.iloc[0:,0:], 
#but don't use df_copy = df (this is a "fake" copy by reference not by value, so changing value in df_copy will change df as well)

#pd.set_option("display.max_rows", None) # show all rows with scrollbar, don't use if there are many rows.
pd.set_option("display.max_rows", 1000) # show all rows with scrollbar, don't use if there are many rows.
pd.set_option("display.max_columns", None)
from IPython.display import display, HTML
CSS = """
.output {
    flex-direction: row;}
"""
HTML('<style>{}</style>'.format(CSS))

from IPython.core.interactiveshell import InteractiveShell
#InteractiveShell.ast_node_interactivity = "all"
#%autosave 180

In [ ]:
def moving_average(f, npoints):
    # n-Point moving average, this window covers the point and its two sides of equal length.
    # The number of N points in moving average must be an odd number.
    f = np.array(f)
    Nsamples = len(f)
    f_ma = copy.deepcopy(f)
    if Nsamples == 0:
        return f_ma
    if npoints <= 0:
        return f_ma
    if npoints % 2 == 0:
        npoints = npoints + 1
    half_length = int((npoints-1)/2)

    for i in range(len(f)):
        if i < half_length and i < Nsamples/2:
            f_ma[i] = sum(f[:(2*i+1)])/(2*i+1)
        elif i < Nsamples - (npoints-1)/2:
            f_ma[i] = sum(f[(i-half_length):(i+half_length+1)])/npoints
        else:
            f_ma[i] = sum(f[(2*i+1-Nsamples):])/(2*Nsamples-2*i-1)

    return f_ma

In [ ]:
def Program_Total_Seconds(t):
    if isinstance(t, str):
        try:
            if t[:-9] == '':
                d = 0
            else:
                d = int(t[:-9])
            h = int(t[-8:-6])
            m = int(t[-5:-3])
            s = int(t[-2:])
            return d*3600*24 + h*3600 + m*60 + s
        except:
            return 0
    elif isinstance(t, datetime.time):
        return t.hour * 3600 + t.minute * 60 + t.second
    else:
        return t

In [ ]:
def Stepsize_Cal(end):
    if end > 30000: stepsize = 2000
    elif end > 15000: stepsize = 1000
    elif end > 7500: stepsize = 500
    elif end > 3000: stepsize = 200
    elif end > 1500: stepsize = 100
    elif end > 750: stepsize = 50
    elif end > 300: stepsize = 20
    elif end > 150: stepsize = 10
    elif end > 75: stepsize = 5
    elif end > 30: stepsize = 2
    else: stepsize = 1
    return stepsize

In [ ]:
SMALL_SIZE = 12
MEDIUM_SIZE = 14
BIG_SIZE = 16
LARGE_SIZE = 20

params = {
    'figure.figsize': (16, 12),
    'font.size': BIG_SIZE,
    'xtick.labelsize': BIG_SIZE,
    'ytick.labelsize': BIG_SIZE,
    'legend.fontsize': BIG_SIZE,
    'figure.titlesize': LARGE_SIZE,
    'axes.titlesize': LARGE_SIZE,
    'axes.labelsize': LARGE_SIZE
}
plt.rcParams.update(params)

In [ ]:
!mkdir figures

## Data Analysis

In [ ]:
ProgramType = '(SCS, DRG, DBS)'
random.seed(1234567)

In [ ]:
Year_Max = 2021
sales_year_range = range(2015, 2020)
predict_year_range = range(2020, Year_Max+1)
display_year_range0 = range(2018, Year_Max+1)
display_year_range = range(2018, Year_Max+5)

In [ ]:
avg_days_adjust = 91 # most patients replace before battery end of life.
avg_days_replace = 464 # average time between implant & explant for non-battery causes, from US complaint data.
battery_replace_ratio = 0.87 # ratio of replaced IPGs due to battery end of life to the total replaced IPGs

### Sales Data Analysis

In [ ]:
dfos = pd.read_excel('Orion.xlsx', sheet_name='Invoice Detail')

In [ ]:
pd.DataFrame({'Data Type':dfos.dtypes, 'Missing Data':dfos.isnull().sum(), 'Unique Values':dfos.nunique(axis=0, dropna=False)})

In [ ]:
dfos.dropna(axis = 0, how = 'all', inplace=True) # drop rows with all NAs
dfos.dropna(axis = 1, how = 'all', inplace=True)
dfos['Time in Quarter'] = dfos.apply(lambda x: str(x['Fiscal Year/Period YYYY0MM'])[:4] + 'Q' + 
                                     str(math.ceil(x['Fiscal Year/Period YYYY0MM']%100/3)), axis=1)
dfos['Time in Half Year'] = dfos.apply(lambda x: str(x['Fiscal Year/Period YYYY0MM'])[:4] + 'H' + 
                                       str(math.ceil(x['Fiscal Year/Period YYYY0MM']%100/6)), axis=1)
dfos = dfos[dfos['GSH - Geography'] == 'DOMESTIC']
dfos['Generator Model'] = dfos['Model - Key'].str.slice(0,4)
IPG_Models = ['3660', '3661', '3662', '3663', '3664', '3665', '3667', '6660', '6661', '6662', '6663']

In [ ]:
dfos = dfos[dfos['Generator Model'].isin(IPG_Models)]

In [ ]:
model_dict = {'3660':'SCS, 5.3Ah Battery', '3661':'SCS, 5.3Ah Battery', '3665':'SCS, 5.3Ah Battery',
              '3662':'SCS, 7.5Ah Battery', '3663':'SCS, 7.5Ah Battery', '3667':'SCS, 7.5Ah Battery',
              '3664':'DRG, 5.3Ah Battery',
              '6660':'DBS, 5.3Ah Battery', '6661':'DBS, 5.3Ah Battery',
              '6662':'DBS, 7.5Ah Battery', '6663':'DBS, 7.5Ah Battery'}
therapy_dict = {'3660':'SCS', '3661':'SCS', '3665':'SCS',
              '3662':'SCS', '3663':'SCS', '3667':'SCS',
              '3664':'DRG',
              '6660':'DBS', '6661':'DBS',
              '6662':'DBS', '6663':'DBS'}

In [ ]:
dfos['Model'] = dfos['Generator Model'].map(model_dict)
dfos['Therapy'] = dfos['Generator Model'].map(therapy_dict)

In [ ]:
[impedance_low, impedance_default, impedance_high] = ['lowest setting', 'default setting', 'highest setting']
if ProgramType == '(7.5Ah Battery)':
    dfos = dfos[dfos['Generator Model'].isin(['3662', '3663', '3667', '6662', '6663'])].reset_index(drop=True)
    ylim_implant = 10500
    ylim_longevity = 1450
    ylim_replacement = 12.5
elif ProgramType == '(5.3Ah Battery)':
    dfos = dfos[dfos['Generator Model'].isin(['3660', '3661', '3664', '3665', '6660', '6661'])].reset_index(drop=True)
    ylim_implant = 10500
    ylim_longevity = 1450
    ylim_replacement = 12.5
elif ProgramType == '(SCS)' or ProgramType == '(Proclaim SCS)' or ProgramType == '(SCS, Proclaim 5 & 7)':
    [impedance_low, impedance_default, impedance_high] = ['200 Ω (Tonic) and 350 Ω (Burst)', '500 Ω', '2000 Ω (Tonic) and 1000 Ω (Burst)']
    dfos = dfos[dfos['Generator Model'].isin(['3660', '3661', '3662', '3663', '3665', '3667'])].reset_index(drop=True)
    ylim_implant = 17000
    ylim_longevity = 3100
    ylim_replacement = 5100
elif ProgramType == '(ProclaimXR)':
    [impedance_low, impedance_default, impedance_high] = ['200 Ω (Tonic) and 350 Ω (Burst)', '500 Ω', '2000 Ω (Tonic) and 1000 Ω (Burst)']
    dfos = dfos[dfos['Generator Model'].isin(['3660', '3662'])].reset_index(drop=True)
    ylim_implant = 17000
    ylim_longevity = 3100
    ylim_replacement = 5100
elif ProgramType == '(Proclaim 5, SCS)' or ProgramType == '(Proclaim 5, SCS, 5.3Ah Battery)':
    [impedance_low, impedance_default, impedance_high] = ['200 Ω (Tonic) and 350 Ω (Burst)', '500 Ω', '2000 Ω (Tonic) and 1000 Ω (Burst)']
    dfos = dfos[dfos['Generator Model'].isin(['3660', '3661', '3665'])].reset_index(drop=True)
    ylim_implant = 8100
    ylim_longevity = 1800
    ylim_replacement = 3100
elif ProgramType == '(Proclaim 7, SCS)' or ProgramType == '(Proclaim 7, SCS, 7.5Ah Battery)':
    [impedance_low, impedance_default, impedance_high] = ['200 Ω (Tonic) and 350 Ω (Burst)', '500 Ω', '2000 Ω (Tonic) and 1000 Ω (Burst)']
    dfos = dfos[dfos['Generator Model'].isin(['3662', '3663', '3667'])].reset_index(drop=True)
    ylim_implant = 8100
    ylim_longevity = 1450
    ylim_replacement = 3100
elif ProgramType == '(SCS Tonic, 5.3Ah Battery)':
    [impedance_low, impedance_default, impedance_high] = ['200 Ω', '500 Ω', '2000 Ω']
    dfos = dfos[dfos['Generator Model'].isin(['3660', '3661', '3665'])].reset_index(drop=True)
    ylim_implant = 8100
    ylim_longevity = 110
    ylim_replacement = 3600
elif ProgramType == '(SCS Tonic, 7.5Ah Battery)':
    [impedance_low, impedance_default, impedance_high] = ['200 Ω', '500 Ω', '2000 Ω']
    dfos = dfos[dfos['Generator Model'].isin(['3662', '3663', '3667'])].reset_index(drop=True)
    ylim_implant = 8100
    ylim_longevity = 200
    ylim_replacement = 12.5
elif ProgramType == '(SCS Burst, 5.3Ah Battery)':
    [impedance_low, impedance_default, impedance_high] = ['350 Ω', '500 Ω', '1000 Ω']
    dfos = dfos[dfos['Generator Model'].isin(['3660', '3661', '3665'])].reset_index(drop=True)
    ylim_implant = 8100
    ylim_longevity = 1050
    ylim_replacement = 12.5
elif ProgramType == '(SCS Burst, 7.5Ah Battery)':
    [impedance_low, impedance_default, impedance_high] = ['350 Ω', '500 Ω', '1000 Ω']
    dfos = dfos[dfos['Generator Model'].isin(['3662', '3663', '3667'])].reset_index(drop=True)
    ylim_implant = 8100
    ylim_longevity = 1050
    ylim_replacement = 12.5
elif ProgramType == '(DRG)' or ProgramType == '(Proclaim DRG)' or ProgramType == '(DRG, 5.3Ah Battery)':
    [impedance_low, impedance_default, impedance_high] = ['200 Ω', '1600 Ω', '4000 Ω']
    dfos = dfos[dfos['Generator Model'].isin(['3664'])].reset_index(drop=True)
    ylim_implant = 1750
    ylim_longevity = 260
    ylim_replacement = 1100
elif ProgramType == '(DBS)' or ProgramType == '(Infinity DBS)':
    [impedance_low, impedance_default, impedance_high] = ['200 Ω', '1200 Ω', '3000 Ω']
    dfos = dfos[dfos['Generator Model'].isin(['6660', '6661', '6662', '6663'])].reset_index(drop=True)
    ylim_implant = 1250
    ylim_longevity = 12
    ylim_replacement = 510
elif ProgramType == '(DBS, 5.3Ah Battery)':
    [impedance_low, impedance_default, impedance_high] = ['200 Ω', '1200 Ω', '3000 Ω']
    dfos = dfos[dfos['Generator Model'].isin(['6660', '6661'])].reset_index(drop=True)
    ylim_implant = 1250
    ylim_longevity = 12
    ylim_replacement = 51
elif ProgramType == '(DBS, 7.5Ah Battery)':
    [impedance_low, impedance_default, impedance_high] = ['200 Ω', '1200 Ω', '3000 Ω']
    dfos = dfos[dfos['Generator Model'].isin(['6662', '6663'])].reset_index(drop=True)
    ylim_implant = 1250
    ylim_longevity = 12
    ylim_replacement = 21
else:
    ylim_implant = 17000
    ylim_longevity = 3100
    ylim_replacement = 6100 * 2

In [ ]:
dfos = dfos[(dfos['Time in Quarter']>='2015Q4') & (dfos['Time in Quarter']<='2020Q3')]

In [ ]:
dfos_r = dfos[dfos['Procedure.1'] == 'Replacement']
#dfos = dfos[dfos['Procedure.1'] == 'DeNovo']
dfos = dfos[dfos['Procedure.1'] != 'Quantity Purchase']

In [ ]:
dfos_hy = dfos.groupby(['Time in Half Year']).agg({'Invoiced Units':'sum'}).reset_index()
dfos_hy.columns = ['Time in Half Year', 'US Total']

In [ ]:
dfos_qt = dfos.groupby(['Time in Quarter']).agg({'Invoiced Units':'sum'}).reset_index()
dfos_qt.columns = ['Time in Quarter', 'US Total']
#dfos_qt['US Total'] = dfos_qt['US Total'].astype(np.int64)

In [ ]:
dfos_hy.at[dfos_hy['Time in Half Year'] == '2020H2','US Total'] = 4 * dfos_hy[dfos_hy['Time in Half Year'] == '2020H2']['US Total']
dfos_qt.at[dfos_qt['Time in Quarter'] == '2020Q3','US Total'] = 2 * dfos_qt[dfos_qt['Time in Quarter'] == '2020Q3']['US Total']

In [ ]:
dfos_qt_scs = dfos[dfos['Therapy']=='SCS'].groupby(['Time in Quarter']).agg({'Invoiced Units':'sum'}).reset_index()
dfos_qt_scs.columns = ['Time in Quarter', 'US Total']
dfos_qt_drg = dfos[dfos['Therapy']=='DRG'].groupby(['Time in Quarter']).agg({'Invoiced Units':'sum'}).reset_index()
dfos_qt_drg.columns = ['Time in Quarter', 'US Total']
dfos_qt_dbs = dfos[dfos['Therapy']=='DBS'].groupby(['Time in Quarter']).agg({'Invoiced Units':'sum'}).reset_index()
dfos_qt_dbs.columns = ['Time in Quarter', 'US Total']

fig,ax = plt.subplots()
fig.set_size_inches(20,10)
plt.grid(axis='y')
ax.plot(dfos_qt['Time in Quarter'], dfos_qt['US Total'], linewidth=2, marker='.', markersize=20, color='red')
ax.plot(dfos_qt_scs['Time in Quarter'], dfos_qt_scs['US Total'], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='blue')
ax.plot(dfos_qt_drg['Time in Quarter'], dfos_qt_drg['US Total'], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='green')
ax.plot(dfos_qt_dbs['Time in Quarter'], dfos_qt_dbs['US Total'], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='purple')

plt.xticks(rotation=45)
ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)

start, end = ax.get_ylim()
end = end * 1.2
stepsize = Stepsize_Cal(end)
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time of Sales (in Quarter)', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)
ax.set_title(('US Orion IPG Sales ' + ProgramType), fontsize=20, fontweight='bold')
lgd = ax.legend(['Total', 'SCS', 'DRG', 'DBS'], loc='best', fontsize=20)
#plt.savefig('figures/' + ProgramType + '_IPG_sales.png')

In [ ]:
from scipy.optimize import curve_fit
from scipy.optimize import least_squares
from scipy.optimize import minimize

xtick = dfos_qt['Time in Quarter'].tolist()
xtick_predict = [(str(Y) + 'Q' + '{}'.format(q)) for Y in predict_year_range for q in range(1, 5)]
xtick = sorted(list(set(dfos_qt['Time in Quarter'].tolist()) | set(xtick_predict)))
fig,ax = plt.subplots()
fig.set_size_inches(20,15)
#ax2 = ax.twinx()

plotx_start = 0
ax.plot(dfos_qt['Time in Quarter'][plotx_start:], dfos_qt['US Total'][plotx_start:], linestyle='--', linewidth=2, marker='.', markersize=20, color='red')

########### Logarithm regression
coef_initial = [1000, 0.1, 4000]
fitting_low_start = 0
bounds = [(100,4000), (-fitting_low_start+0.1,10), (-10000,10000)]
bounds1 = ([100, -fitting_low_start+0.1, -10000], [4000, 10, 10000])
x = np.arange(fitting_low_start,len(dfos_qt['Time in Quarter']))
y = dfos_qt['US Total'][fitting_low_start:]

def func(x,*p):
    a,b,c = p
    return a*np.log(x+b) + c

# # method 1, https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html
# def func1(x, a, b, c):
#     return a*np.log(x+b) + c
# coef = sp.optimize.curve_fit(func1, x, y,
#                              #method='lm',
#                              bounds=bounds1)
# coefs = coef[0]

# # method 2, https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html, https://scipy-cookbook.readthedocs.io/items/robust_regression.html
def func2(coef, x, y):
    return coef[0]*np.log(x+coef[1]) + coef[2] - y
coef = least_squares(func2, coef_initial, loss='linear', # 'linear', 'huber', 'soft_l1', 'cauchy', 'arctan', 
                     bounds = bounds1,
                     f_scale=0.1, args=(x, y))
coefs = coef.x

# # method 3, https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html
# def func3(x,*p):
#     a1,a2,a3 = p
#     return a1*np.log(x+a2) + a3
# err = lambda p: np.mean((func3(x,*p)-y)**2)
# coef = minimize(err, coef_initial,
#                 bounds = bounds,
#                 method = "L-BFGS-B")
# coefs = coef.x

print(*coefs)
sales_predict_qt_low = func(np.arange(fitting_low_start, len(xtick)), *coefs).round().astype(int); print(sales_predict_qt_low)
ax.plot(xtick[fitting_low_start:], sales_predict_qt_low, marker='.', markersize=10, color='blue')

############

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)

start, end = ax.get_ylim()
ylim_sales = end * 1.3
ax.set(ylim=[0, ylim_sales])
start, end = ax.get_ylim()
stepsize = Stepsize_Cal(end)
ax.yaxis.set_ticks(np.arange(start, end, stepsize))
ax.set_xlabel('Time of Sales (in Quarter)', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

# ax2.plot(dfos_qt['Time in Quarter'][plotx_start:], dfos_qt['US Total'][plotx_start:], linestyle='--', linewidth=2, marker='.', markersize=20, color='red')
# ax2.plot(xtick[fitting_low_start:], sales_predict_qt_low, marker='.', markersize=10, color='blue')
# ax2.tick_params(axis='y', labelsize=20, labelcolor='red')
# ax2.set(ylim=[0, ylim_sales])
# start, end = ax2.get_ylim()    
# stepsize = Stepsize_Cal(end)
# ax2.yaxis.set_ticks(np.arange(start, end, stepsize))
# ax2.set_ylabel('Orion IPG Counts', fontsize=20, color='red')

ax.grid(axis='y')
ax.legend(['Actual Sales','Predicted Sales\n(logarithm regression model)'], loc='lower right', fontsize=20)
ax.set_title(('US Orion IPG Sales & Trend Forecast ' + ProgramType), fontsize=20, fontweight='bold')
#locs, labels = plt.xticks()
#plt.setp(labels, rotation=90)
#ax.set_xticklabels(labels, rotation=90)
#plt.xticks(locs, labels, rotation='vertical')
for tick in ax.get_xticklabels():
    tick.set_rotation(90)
plt.tick_params(labelright=True)
#plt.savefig('figures/' + ProgramType + '_IPG_sales_predict_qt.png')

In [ ]:
dfos_hy = dfos_hy[:-1]
dfos_qt = dfos_qt[:-1]

In [ ]:
index_hy = sorted(list(set([(str(Y) + 'H' + '{}'.format(q)) for Y in predict_year_range for q in range(1, 3)]) -
                       set(dfos_hy['Time in Half Year'].tolist())))
dfos_predict_hy_low = pd.DataFrame({'Time in Half Year': index_hy,
                                'US Total': sales_predict_qt_low[-len(index_hy)*2::2] + sales_predict_qt_low[(-len(index_hy)*2+1)::2]})
index_qt = sorted(list(set([(str(Y) + 'Q' + '{}'.format(q)) for Y in predict_year_range for q in range(1, 5)]) -
                       set(dfos_qt['Time in Quarter'].tolist())))
dfos_predict_qt_low = pd.DataFrame({'Time in Quarter': index_qt, 
                                'US Total': sales_predict_qt_low[-len(index_qt):]})

In [ ]:
if ProgramType == '(SCS, DRG, DBS)':
    dfus_model = dfos.groupby(['Therapy']).agg({'Invoiced Units':'sum'}).reset_index()

    labels = dfus_model['Therapy']
    sizes = dfus_model['Invoiced Units']
    
    sort_legend = True
    if sort_legend:
        sizes, labels =  zip(*sorted(zip(sizes, labels), key=lambda x: x[1], reverse=True))
        
    labels = ['{0} ({1} sales) - {2:1.2f} % of total'.format(i,j,k) for i,j,k in zip(labels, sizes, sizes/dfus_model['Invoiced Units'].sum()*100)]

    fig, ax = plt.subplots()
    fig.set_size_inches(8,8)
    # patches, texts, autotexts = ax.pie(sizes, colors = colors, labels=labels, explode=explode, #startangle=90, 
    #                                   textprops={'fontsize': 18}, autopct='%1.1f%%', pctdistance=0.8, labeldistance=1.05)
    patches, texts = ax.pie(sizes)
    for text in texts:
        text.set_color('black')
        #text.set_fontsize(18)
    #for autotext in autotexts:
    #    #autotext.set_color('grey')
    #    autotext.set_fontsize(16)
    # Equal aspect ratio ensures that pie is drawn as a circle
    ax.axis('equal')  
    ax.set_title(r'$\bf{' + f"US\ Orion\ IPG\ Sales" + '}$' + '\n' +
                 '-- by therapy type (2015~2020, total = {} sales)'.format(dfus_model['Invoiced Units'].sum()), fontsize=20)
    lgd = ax.legend(patches, labels, loc='best', bbox_to_anchor=(1.1, 1), fontsize=18)
    #plt.tight_layout()
    plt.savefig('figures/' + ProgramType + '_IPG_sales_therapy_pie.png', bbox_inches='tight') #, bbox_extra_artist=[lgd])

In [ ]:
if ProgramType == '(SCS, DRG, DBS)':
    dfus_model = dfos.groupby(['Model']).agg({'Invoiced Units':'sum'}).reset_index()
    #dfus_model.sort_values('Model', ascending=False, inplace=True)
    labels = dfus_model['Model']
    sizes = dfus_model['Invoiced Units']/dfus_model['Invoiced Units'].sum()*100
    
    sort_legend = True
    if sort_legend:
        sizes, labels =  zip(*sorted(zip(sizes, labels), key=lambda x: x[0], reverse=True))
        
    labels = ['{0} - {1:1.2f} %'.format(i,j) for i,j in zip(labels, sizes)]

    #colors
    colors = [(0,0.8,0,0.8), (0,0,0.8,0.8), (1,1,0,0.8), (1,0,0,0.8), (0,0,0,1)]
    explode = [0, 0, 0.05, 0.10, 0.10]

    fig, ax = plt.subplots()
    fig.set_size_inches(8,8)
    patches, texts = ax.pie(sizes, colors = colors, explode=explode)
    # patches, texts, autotexts = ax.pie(sizes, colors = colors, labels=labels, explode=explode, #startangle=90, 
    #                                   textprops={'fontsize': 18}, autopct='%1.1f%%', pctdistance=0.8, labeldistance=1.05)
    #for text in texts:
    #    text.set_color('black')
    #    #text.set_fontsize(18)
    #for autotext in autotexts:
    #    #autotext.set_color('grey')
    #    autotext.set_fontsize(16)
    
    # Equal aspect ratio ensures that pie is drawn as a circle
    ax.axis('equal')  
    ax.set_title(r'$\bf{' + f"US\ Orion\ IPG\ Sales" + '}$' + '\n' +
                 '(2016~2020, total = {})'.format(dfus_model['Invoiced Units'].sum()), fontsize=20)
    lgd = ax.legend(patches, labels, loc='best', bbox_to_anchor=(1.1, 1), fontsize=18)
    #plt.tight_layout()
    #plt.savefig('figures/' + ProgramType + '_IPG_sales_pie.png', bbox_inches='tight') #, bbox_extra_artist=[lgd])

In [ ]:
if ProgramType == '(SCS, DRG, DBS)':
    dfus_model = dfos.groupby(['Generator Model']).agg({'Invoiced Units':'sum'}).reset_index()

    labels = dfus_model['Generator Model']
    sizes = dfus_model['Invoiced Units']/dfus_model['Invoiced Units'].sum()*100
    
    sort_legend = True
    if sort_legend:
        sizes, labels =  zip(*sorted(zip(sizes, labels), key=lambda x: x[1], reverse=False))
        
    labels = [model_dict[i] + ', Model {0} - {1:1.2f} %'.format(i,j) for i,j in zip(labels, sizes)]

    #colors
    colors = [(0,1,0,0.8), (0,0.5,0,0.8),
              (0,1,1,0.8), (0,0,0.7,0.8), 
              (1,1,0,0.8), 
              (1,0,0,0.8), (0.5,0,0,0.8),
              (0,0,0,1), (0,0,0,0.5)]
    explode = [0.05, 0,
               0, 0,
               0.05, 
               0.10, 0.10,
               0.10, 0.10]

    fig, ax = plt.subplots()
    fig.set_size_inches(8,8)
    patches, texts = ax.pie(sizes, colors = colors, explode=explode)
    # patches, texts, autotexts = ax.pie(sizes, colors = colors, labels=labels, explode=explode, #startangle=90, 
    #                                   textprops={'fontsize': 18}, autopct='%1.1f%%', pctdistance=0.8, labeldistance=1.05)
    #for text in texts:
    #    text.set_color('black')
    #    #text.set_fontsize(18)
    #for autotext in autotexts:
    #    #autotext.set_color('grey')
    #    autotext.set_fontsize(16)

    # Equal aspect ratio ensures that pie is drawn as a circle
    ax.axis('equal')  
    ax.set_title(r'$\bf{' + f"US\ Orion\ IPG\ Sales" + '}$' + '\n' +
                 '-- by model number (2015~2020, total = {})'.format(dfus_model['Invoiced Units'].sum()), fontsize=20)
    lgd = ax.legend(patches, labels, loc='best', bbox_to_anchor=(1.1, 1), fontsize=18)
    #plt.tight_layout()
    #plt.savefig('figures/' + ProgramType + '_IPG_sales_model_pie.png', bbox_inches='tight') #, bbox_extra_artist=[lgd])

### CP Data Analysis

In [ ]:
CP_datatype = 'database' # 'log' or 'database'

In [ ]:
if CP_datatype.lower() == 'log':
    df = pd.read_csv('CP_Log_data_withlongevity_2015-2020.xls', sep='\t')
if CP_datatype.lower() == 'database':
    df = pd.read_csv('CP_Database_data_withLongevity_2015-2020.xls', sep='\t')
    df.rename(columns={'SerialNumber':'Serial Number (hashed)',
                       'Model':'Generator Model'}, inplace=True) 

In [ ]:
df['Generator Model'] = df['Generator Model'].astype(str)
IPG_Models = ['3660', '3661', '3662', '3663', '3664', '3665', '3667', '6660', '6661', '6662', '6663']
df = df[df['Generator Model'].isin(IPG_Models)]

In [ ]:
if CP_datatype.lower() == 'log':
    df = df[df['Generator Model'] != 'System.Byte[]']
    df = df[df['Generator Model'].notna()]
    df = df[df['Session Active Program'] != 16]
    df = df[df['Program Type'].isin(['Tonic', 'Burst', 'DBS', 'DbsDual', 'DRG'])]
    df = df[df['Running Mode'] != 'Off']
    df = df[df['Electrodes']!='="0000000000000000"']
    df = df[(df[['Longevity_low','Longevity_medium','Longevity_high']] != -1).all(axis=1)].reset_index(drop=True)

    #df['Implant Date'] = df[['Implant Date', 'Manufacture Date']].max(axis=1)
    df = df[df['Implant Date'] >= df['Manufacture Date']]

    if ProgramType == '(7.5Ah Battery)':
        df = df[df['Generator Model'].isin(['3662', '3663', '3667', '6662', '6663'])].reset_index(drop=True)
    elif ProgramType == '(5.3Ah Battery)':
        df = df[df['Generator Model'].isin(['3660', '3661', '3664', '3665', '6660', '6661'])].reset_index(drop=True)
    elif ProgramType == '(SCS)' or ProgramType == '(Proclaim SCS)' or ProgramType == '(SCS, Proclaim 5 & 7)':
        df = df[df['Generator Model'].isin(['3660', '3661', '3662', '3663', '3665', '3667'])].reset_index(drop=True)
    elif ProgramType == '(ProclaimXR)':
        df = df[df['Generator Model'].isin(['3660', '3662'])].reset_index(drop=True)
    elif ProgramType == '(Proclaim 5, SCS)' or ProgramType == '(Proclaim 5, SCS, 5.3Ah Battery)':
        df = df[df['Generator Model'].isin(['3660', '3661', '3665'])].reset_index(drop=True)
    elif ProgramType == '(Proclaim 7, SCS)' or ProgramType == '(Proclaim 7, SCS, 7.5Ah Battery)':
        df = df[df['Generator Model'].isin(['3662', '3663', '3667'])].reset_index(drop=True)
    elif ProgramType == '(SCS Tonic, 5.3Ah Battery)':
        df = df[df['Generator Model'].isin(['3660', '3661', '3665'])].reset_index(drop=True)
        df = df[df['Program Type'] == 'Tonic'].reset_index(drop=True)
    elif ProgramType == '(SCS Tonic, 7.5Ah Battery)':
        df = df[df['Generator Model'].isin(['3662', '3663', '3667'])].reset_index(drop=True)
        df = df[df['Program Type'] == 'Tonic'].reset_index(drop=True)
    elif ProgramType == '(SCS Burst, 5.3Ah Battery)':
        df = df[df['Generator Model'].isin(['3660', '3661', '3665'])].reset_index(drop=True)
        df = df[df['Program Type'] == 'Burst'].reset_index(drop=True)
    elif ProgramType == '(SCS Burst, 7.5Ah Battery)':
        df = df[df['Generator Model'].isin(['3662', '3663', '3667'])].reset_index(drop=True)
        df = df[df['Program Type'] == 'Burst'].reset_index(drop=True)
    elif ProgramType == '(DRG)' or ProgramType == '(Proclaim DRG)' or ProgramType == '(DRG, 5.3Ah Battery)':
        df = df[df['Generator Model'].isin(['3664'])].reset_index(drop=True)
    elif ProgramType == '(DBS)' or ProgramType == '(Infinity DBS)':
        df = df[df['Generator Model'].isin(['6660', '6661', '6662', '6663'])].reset_index(drop=True)
    elif ProgramType == '(DBS, 5.3Ah Battery)':
        df = df[df['Generator Model'].isin(['6660', '6661'])].reset_index(drop=True)
    elif ProgramType == '(DBS, 7.5Ah Battery)':
        df = df[df['Generator Model'].isin(['6662', '6663'])].reset_index(drop=True)

    # Pick session that has latest date
    df = df[df['Serial Number (hashed)'].map(df.groupby(['Serial Number (hashed)'])['Log Date'].max()) == df['Log Date']]
    #df = df[df.groupby(['Serial Number (hashed)'])['Log Date'].transform(max) == df['Log Date']]

    # Pick program that has maximum Program On Time
    df['Program On Time'] = df['Program On Time'].apply(lambda x: Program_Total_Seconds(x))
    df = df[df['Serial Number (hashed)'].map(df.groupby(['Serial Number (hashed)'])['Program On Time'].max()) == df['Program On Time']]
    # Pick program with lowest longevity to represent IPG's working program
    df = df[df['Serial Number (hashed)'].map(df.groupby(['Serial Number (hashed)'])['Longevity_medium'].min()) == df['Longevity_medium']]
    # Pick Burst if Tonic and Burst have equal counts (in string format, Burst < Tonic) 
    df = df[df['Serial Number (hashed)'].map(df.groupby(['Serial Number (hashed)'])['Program Type'].min()) == df['Program Type']]
    # Pick Cycle if Cycle and Continuous have equal counts (in string format, Cycle > Continuous) 
    df = df[df['Serial Number (hashed)'].map(df.groupby(['Serial Number (hashed)'])['Running Mode'].max()) == df['Running Mode']]

    # show rows with duplicates
    #df[df.duplicated('Serial Number (hashed)', keep=False)].sort_values('Serial Number (hashed)').reset_index(drop=True);

    df['Cycle On Time'] = df['Cycle On Time'].fillna('Continuous')
    df['Cycle Off Time'] = df['Cycle Off Time'].fillna('Continuous')

    # get mode
    df_tmp = df.groupby('Serial Number (hashed)')['Therapy', 'Generator Model', 'Log Date', 'Program Type', 'Program On Time', 'Running Mode', 'Cycle On Time', 'Cycle Off Time'].agg(lambda x:x.value_counts().index[0]).reset_index()

    # # Calculate mean amplitude for each program (each program may have multiple stimsets),
    # # then calculate mean amplitude of all programs of each IPG
#     df['Program ID'] = df['Log File Name'] + df['Program Name']
#     df = df.groupby(['Program ID']).agg({'Serial Number (hashed)':['min'], 'Implant Date':['min'], 'Longevity_low':[stats.hmean], 'Longevity_medium':[stats.hmean],
#                                         'Longevity_high':[stats.hmean]}).reset_index()
#     df.columns = ['Program ID', 'Serial Number (hashed)', 'Implant Date', 'Longevity_low', 'Longevity_medium', 'Longevity_high'] 

    df = df.groupby(['Serial Number (hashed)']).agg({'Implant Date':['min'], 'Longevity_low':[stats.hmean], 'Longevity_medium':[stats.hmean],
                                                     'Longevity_high':[stats.hmean]}).reset_index()
    #df = df.groupby(['Serial Number (hashed)']).agg({'Implant Date':['min'], 'Longevity_low':['min'], 'Longevity_medium':[stats.hmean],
    #                                           'Longevity_high':['max']}).reset_index()
    df.columns = ['Serial Number (hashed)', 'Implant Date', 'Longevity_low', 'Longevity_medium', 'Longevity_high'] 

    #def f(x1, x2):
    #    return datetime.datetime.strptime(x1, "%Y/%m/%d %H:%M:%S") + timedelta(days=365*x2)
    #df['Replacement Date Earliest'] = df.apply(lambda x: f(x['Implant Date'], x['Longevity_low']), axis=1)
    #df['Replacement Date Earliest'] = df['Implant Date'].apply(lambda x : datetime.datetime.strptime(x, "%Y/%m/%d %H:%M:%S")) + df['Longevity_low'].apply(lambda x: timedelta(days=365*x))
    df['Implant Date'] = df.apply(lambda x: datetime.strptime(x['Implant Date'], "%Y/%m/%d %H:%M:%S").date(), axis=1)
    df['Replacement Date Earliest'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_low']-avg_days_adjust), axis=1)
    df['Replacement Date Average'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_medium']-avg_days_adjust), axis=1)
    df['Replacement Date Latest'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_high']-avg_days_adjust), axis=1)

In [ ]:
if CP_datatype.lower() == 'database':
    df = df[df['Generator Model'].notna()]
    df = df[df['ProgramType'].isin([0, 1, 2, 5, 6])]
    df = df[df['RunningMode'] != 0]
    df = df[df['Electrodes']!='="0000000000000000"']
    df = df[(df[['Longevity_low','Longevity_medium','Longevity_high']] != -1).all(axis=1)].reset_index(drop=True)

    df['Implant Date'] = df['InitialSessionDate']

    if ProgramType == '(7.5Ah Battery)':
        df = df[df['Generator Model'].isin(['3662', '3663', '3667', '6662', '6663'])].reset_index(drop=True)
    elif ProgramType == '(5.3Ah Battery)':
        df = df[df['Generator Model'].isin(['3660', '3661', '3664', '3665', '6660', '6661'])].reset_index(drop=True)
    elif ProgramType == '(SCS)' or ProgramType == '(Proclaim SCS)' or ProgramType == '(SCS, Proclaim 5 & 7)':
        df = df[df['Generator Model'].isin(['3660', '3661', '3662', '3663', '3665', '3667'])].reset_index(drop=True)
    elif ProgramType == '(ProclaimXR)':
        df = df[df['Generator Model'].isin(['3660', '3662'])].reset_index(drop=True)
    elif ProgramType == '(Proclaim 5, SCS)' or ProgramType == '(Proclaim 5, SCS, 5.3Ah Battery)':
        df = df[df['Generator Model'].isin(['3660', '3661', '3665'])].reset_index(drop=True)
    elif ProgramType == '(Proclaim 7, SCS)' or ProgramType == '(Proclaim 7, SCS, 7.5Ah Battery)':
        df = df[df['Generator Model'].isin(['3662', '3663', '3667'])].reset_index(drop=True)
    elif ProgramType == '(SCS Tonic, 5.3Ah Battery)':
        df = df[df['Generator Model'].isin(['3660', '3661', '3665'])].reset_index(drop=True)
        df = df[df['ProgramType'] == 0].reset_index(drop=True)
    elif ProgramType == '(SCS Tonic, 7.5Ah Battery)':
        df = df[df['Generator Model'].isin(['3662', '3663', '3667'])].reset_index(drop=True)
        df = df[df['ProgramType'] == 0].reset_index(drop=True)
    elif ProgramType == '(SCS Burst, 5.3Ah Battery)':
        df = df[df['Generator Model'].isin(['3660', '3661', '3665'])].reset_index(drop=True)
        df = df[df['ProgramType'] == 1].reset_index(drop=True)
    elif ProgramType == '(SCS Burst, 7.5Ah Battery)':
        df = df[df['Generator Model'].isin(['3662', '3663', '3667'])].reset_index(drop=True)
        df = df[df['ProgramType'] == 1].reset_index(drop=True)
    elif ProgramType == '(DRG)' or ProgramType == '(Proclaim DRG)' or ProgramType == '(DRG, 5.3Ah Battery)':
        df = df[df['Generator Model'].isin(['3664'])].reset_index(drop=True)
    elif ProgramType == '(DBS)' or ProgramType == '(Infinity DBS)':
        df = df[df['Generator Model'].isin(['6660', '6661', '6662', '6663'])].reset_index(drop=True)
    elif ProgramType == '(DBS, 5.3Ah Battery)':
        df = df[df['Generator Model'].isin(['6660', '6661'])].reset_index(drop=True)
    elif ProgramType == '(DBS, 7.5Ah Battery)':
        df = df[df['Generator Model'].isin(['6662', '6663'])].reset_index(drop=True)

    # Pick session that has latest date        
    df = df[df['Serial Number (hashed)'].map(df.groupby(['Serial Number (hashed)'])['StartSessionDate'].max()) == df['StartSessionDate']]
    #df = df[df.groupby(['Serial Number (hashed)'])['StartSessionDate'].transform(max) == df['StartSessionDate']]

    # Pick program that has maximum ProgramOnTime
    df = df[df['Serial Number (hashed)'].map(df.groupby(['Serial Number (hashed)'])['ProgramOnTime'].max()) == df['ProgramOnTime']]
    # Pick program with lowest longevity to represent IPG's working program
    df = df[df['Serial Number (hashed)'].map(df.groupby(['Serial Number (hashed)'])['Longevity_medium'].min()) == df['Longevity_medium']]
    # Pick 1:Burst if 0:Tonic and 1:Burst have equal counts (in enum format, 1:Burst > 0:Tonic) 
    df = df[df['Serial Number (hashed)'].map(df.groupby(['Serial Number (hashed)'])['ProgramType'].max()) == df['ProgramType']]
    # Pick 2:Cycle if 2:Cycle and 3:Continuous have equal counts (in enum format, 2:Cycle < 3:Continuous) 
    df = df[df['Serial Number (hashed)'].map(df.groupby(['Serial Number (hashed)'])['RunningMode'].min()) == df['RunningMode']]

    # show rows with duplicates
    #df[df.duplicated('Serial Number (hashed)', keep=False)].sort_values('Serial Number (hashed)').reset_index(drop=True);

    df['RunningModeOn_sec'] = df['RunningModeOn_sec'].fillna(0)
    df['RunningModeOff_sec'] = df['RunningModeOff_sec'].fillna(0)

    # get mode
    df_tmp = df.groupby('Serial Number (hashed)')['Therapy', 'Generator Model', 'StartSessionDate', 'ProgramType', 'ProgramOnTime', 'RunningMode', 'RunningModeOn_sec', 'RunningModeOff_sec'].agg(lambda x:x.value_counts().index[0]).reset_index()

    # # Calculate mean amplitude for each program (each program may have multiple stimsets),
    # # then calculate mean amplitude of all programs of each IPG
#     df = df.groupby(['ProgramId']).agg({'Serial Number (hashed)':['min'], 'Implant Date':['min'], 'Longevity_low':[stats.hmean], 'Longevity_medium':[stats.hmean],
#                                          'Longevity_high':[stats.hmean]}).reset_index()
#     df.columns = ['ProgramId', 'Serial Number (hashed)', 'Implant Date', 'Longevity_low', 'Longevity_medium', 'Longevity_high'] 

    df = df.groupby(['Serial Number (hashed)']).agg({'Implant Date':['min'], 'Longevity_low':[stats.hmean], 'Longevity_medium':[stats.hmean],
                                                     'Longevity_high':[stats.hmean]}).reset_index()
    #df = df.groupby(['Serial Number (hashed)']).agg({'Implant Date':['min'], 'Longevity_low':['min'], 'Longevity_medium':[stats.hmean],
    #                                           'Longevity_high':['max']}).reset_index()
    df.columns = ['Serial Number (hashed)', 'Implant Date', 'Longevity_low', 'Longevity_medium', 'Longevity_high'] 

    #def f(x1, x2):
    #    return datetime.datetime.strptime(x1, "%m/%d/%Y %H:%M:%S") + timedelta(days=365*x2)
    #df['Replacement Date Earliest'] = df.apply(lambda x: f(x['Date'], x['Longevity_low']), axis=1)
    #df['Replacement Date Earliest'] = df['Date'].apply(lambda x : datetime.datetime.strptime(x, "%m/%d/%Y %H:%M:%S")) + df['Longevity_low'].apply(lambda x: timedelta(days=365*x))
    df['Implant Date'] = df.apply(lambda x: datetime.strptime(x['Implant Date'], "%m/%d/%Y %H:%M:%S").date(), axis=1)
    df['Replacement Date Earliest'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_low']-avg_days_adjust), axis=1)
    df['Replacement Date Average'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_medium']-avg_days_adjust), axis=1)
    df['Replacement Date Latest'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_high']-avg_days_adjust), axis=1)

In [ ]:
df['Implant Time'] = df['Implant Date'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
df['Replacement Earliest'] = df['Replacement Date Earliest'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
df['Replacement Average'] = df['Replacement Date Average'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
df['Replacement Latest'] = df['Replacement Date Latest'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
df['Implant Time Quarter'] = df['Implant Date'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
df['Replacement Earliest Quarter'] = df['Replacement Date Earliest'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
df['Replacement Average Quarter'] = df['Replacement Date Average'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
df['Replacement Latest Quarter'] = df['Replacement Date Latest'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))

In [ ]:
df = df[(df['Implant Time']>='2015') & (df['Implant Time']<'2021')]
#df = copy.deepcopy(df[(df['Implant Time']>='2015') & (df['Implant Time']<'2021')])
#df = copy.deepcopy(df[(df['Implant Time Quarter']>='2015') & (df['Implant Time Quarter']<'2021')])

In [ ]:
df = pd.merge(df_tmp, df, on='Serial Number (hashed)', how='inner').reset_index(drop=True)

In [ ]:
if ProgramType == '(SCS, DRG, DBS)':
    df_model = df.groupby(['Therapy']).agg({'Serial Number (hashed)':'count'}).reset_index().rename(columns={'Serial Number (hashed)':'Count'})
    labels = df_model['Therapy']
    sizes = df_model['Count']
    
    if CP_datatype.lower() == 'database':
        therapy_dict = {0:'SCS', 1:'DBS', 2:'DRG'}
        labels = labels.map(therapy_dict)
        
    sort_legend = True
    if sort_legend:
        sizes, labels =  zip(*sorted(zip(sizes, labels), key=lambda x: x[1], reverse=True))
        
    labels = ['{0} ({1} IPGs) - {2:1.2f} % of total'.format(i,j,k) for i,j,k in zip(labels, sizes, sizes/df_model['Count'].sum()*100)]

    fig, ax = plt.subplots()
    fig.set_size_inches(8,8)
    patches, texts = ax.pie(sizes)

    ax.axis('equal')  
    ax.set_title(r'$\bf{' + f"US\ Orion\ IPGs\ Found\ in\ Data" + '}$' + '\n' +
                 '-- by model number (2015~2020, total = {} IPGs)'.format(df_model['Count'].sum()), fontsize=20)
    ax.legend(patches, labels, loc='best', bbox_to_anchor=(1.1, 1), fontsize=18)
    plt.savefig('figures/' + ProgramType + '_IPG_found_in_data_therapy_pie.png', bbox_inches='tight')

In [ ]:
if ProgramType == '(SCS, DRG, DBS)':
    if CP_datatype.lower() == 'log':
        df_type = df.groupby(['Program Type', 'Running Mode']).agg({'Serial Number (hashed)':'count'}).reset_index().rename(columns={'Serial Number (hashed)':'Count'})
        df_type['Program Type'] = df_type['Program Type'].map({'Burst':'SCS Burst', 'Tonic':'SCS Tonic', 'DRG':'DRG', 'DBS':'DBS', 'DbsDual':'DBSDual'}) # .replace('Burst', 'SCS Burst')
        labels = df_type['Program Type'] + ' ' + df_type['Running Mode']
    if CP_datatype.lower() == 'database':
        df_type = df.groupby(['ProgramType', 'RunningMode']).agg({'Serial Number (hashed)':'count'}).reset_index().rename(columns={'Serial Number (hashed)':'Count'})
        df_type['ProgramType'] = df_type['ProgramType'].map({0:'SCS Tonic', 1:'SCS Burst', 2:'DBS', 5:'DBS', 6:'DRG'})
        df_type['RunningMode'] = df_type['RunningMode'].map({0:'Off', 1:'Sleep', 2:'Cycle', 3:'Continuous'})
        labels = df_type['ProgramType'] + ' ' + df_type['RunningMode']

    sizes = df_type['Count']/df_type['Count'].sum()*100
    
    sort_legend = True
    if sort_legend:
        sizes, labels =  zip(*sorted(zip(sizes, labels), key=lambda x: x[1], reverse=True))

    labels = ['{0} - {1:1.2f} %'.format(i,j) for i,j in zip(labels, sizes)]

    #colors
#     colors = [(0,1,0,0.8), (0,1,1,0.8),
#               (0,0.5,0,0.8), (0,0,0.7,0.8), 
#               (0,0,0,1), (1,1,0,0.8), 
#               (1,0,0,0.8)]
#     explode = [0, 0,
#                0, 0,
#                0.05, 0.05,
#                0.10]

    fig, ax = plt.subplots()
    fig.set_size_inches(8,8)
    # patches, texts = ax.pie(sizes, colors = colors, explode=explode)
    patches, texts = ax.pie(sizes)

    ax.axis('equal')  
    ax.set_title(r'$\bf{' + f"US\ Orion\ IPGs\ Found\ in\ Data" + '}$' + '\n' +
                 '-- by therapy & mode (2015~2020, total = {})'.format(df_type['Count'].sum()), fontsize=20)
    ax.legend(patches, labels, loc='best', bbox_to_anchor=(1.1, 1), fontsize=18)
    #plt.savefig('figures/' + ProgramType + '_IPG_found_in_data_Mode_pie.png', bbox_inches='tight')

In [ ]:
if ProgramType == '(SCS, DRG, DBS)':
    if CP_datatype.lower() == 'log':
        df_scs_burst = df[(df['Program Type']=='Burst') & (df['Running Mode']=='Cycle')]
        df_cycle = df_scs_burst.groupby(['Cycle On Time', 'Cycle Off Time']).agg({'Serial Number (hashed)':'count'}).reset_index().rename(columns={'Serial Number (hashed)':'Count'})
        df_cycle.sort_values('Count', ascending=False, inplace=True)
        labels = list('Cycle On ' + df_cycle['Cycle On Time'][:9] + ', Cycle Off ' + df_cycle['Cycle Off Time'][:9]) + ['All Other Minors Summed']
    if CP_datatype.lower() == 'database':
        df_scs_burst = df[(df['ProgramType']==1) & (df['RunningMode']==2)]
        df_cycle = df_scs_burst.groupby(['RunningModeOn_sec', 'RunningModeOff_sec']).agg({'Serial Number (hashed)':'count'}).reset_index().rename(columns={'Serial Number (hashed)':'Count'})
        df_cycle.sort_values('Count', ascending=False, inplace=True)
        labels = list('Cycle On ' + df_cycle['RunningModeOn_sec'][:9].astype(str) + ' sec, Cycle Off ' + df_cycle['RunningModeOff_sec'][:9].astype(str) + ' sec') + ['All Other Minors Summed']
    sizes = (df_cycle['Count'][:9]/df_cycle['Count'].sum()*100).tolist() + [100 - sum(df_cycle['Count'][:9]/df_cycle['Count'].sum()*100)]
    labels = ['{0} - {1:1.2f} %'.format(i,j) for i,j in zip(labels, sizes)]

    fig, ax = plt.subplots()
    fig.set_size_inches(8,8)
    patches, texts = ax.pie(sizes)

    ax.axis('equal')  
    ax.set_title(r'$\bf{' + f"US\ Orion\ IPGs\ Found\ in\ Data\ and\ in\ Burst\ Mode" + '}$' + '\n' +
                 '-- by cycle setting (2016~2020, total = {})'.format(df_cycle['Count'].sum()), fontsize=20)
    ax.legend(patches, labels, loc='best', bbox_to_anchor=(1.1, 1), fontsize=18)
    #plt.savefig('figures/' + 'SCS_Burst' + '_IPG_found_in_data_pie.png', bbox_inches='tight')

In [ ]:
if ProgramType == '(SCS, DRG, DBS)':
    df['Model'] = df['Generator Model'].map(model_dict)
    df_model = df.groupby(['Model']).agg({'Serial Number (hashed)':'count'}).reset_index().rename(columns={'Serial Number (hashed)':'Count'})
    labels = df_model['Model']
    sizes = df_model['Count']/df_model['Count'].sum()*100
    
    sort_legend = True
    if sort_legend:
        sizes, labels =  zip(*sorted(zip(sizes, labels), key=lambda x: x[1], reverse=True))
        
    labels = ['{0} - {1:1.2f} %'.format(i,j) for i,j in zip(labels, sizes)]

    #colors
    colors = [(0,0.8,0,0.8), (0,0,0.8,0.8), (1,1,0,0.8), (1,0,0,0.8), (0,0,0,1)]
    explode = [0, 0, 0.05, 0.10, 0.10]

    fig, ax = plt.subplots()
    fig.set_size_inches(8,8)
    patches, texts = ax.pie(sizes, colors = colors, explode=explode)

    ax.axis('equal')  
    ax.set_title(r'$\bf{' + f"US\ Orion\ IPGs\ Found\ in\ Data" + '}$' + '\n' +
                 '-- by therapy & battery (2015~2020, total = {})'.format(df_model['Count'].sum()), fontsize=20)
    ax.legend(patches, labels, loc='best', bbox_to_anchor=(1.1, 1), fontsize=18)
    #plt.savefig('figures/' + ProgramType + '_IPG_found_in_data_pie.png', bbox_inches='tight')

In [ ]:
if ProgramType == '(SCS, DRG, DBS)':
    df_model = df.groupby(['Generator Model']).agg({'Serial Number (hashed)':'count'}).reset_index().rename(columns={'Serial Number (hashed)':'Count'})
    labels = df_model['Generator Model']
    sizes = df_model['Count']/df_model['Count'].sum()*100
    
    sort_legend = True
    if sort_legend:
        sizes, labels =  zip(*sorted(zip(sizes, labels), key=lambda x: x[0], reverse=True))
    labels = [model_dict[i] + ', Model {0} - {1:1.2f} %'.format(i,j) for i,j in zip(labels, sizes)]

    #colors
    colors = [(0,1,0,0.8), (0,0.5,0,0.8),
              (0,1,1,0.8), (0,0,0.7,0.8), 
              (1,1,0,0.8), 
              (1,0,0,0.8), (0.5,0,0,0.8),
              (0,0,0,1), (0,0,0,0.5)]
    explode = [0, 0,
               0, 0,
               0.05, 
               0.10, 0.10,
               0.10, 0.10]

    fig, ax = plt.subplots()
    fig.set_size_inches(8,8)
    patches, texts = ax.pie(sizes, colors = colors, explode=explode)

    ax.axis('equal')  
    ax.set_title(r'$\bf{' + f"US\ Orion\ IPGs\ Found\ in\ Data" + '}$' + '\n' +
                 '-- by model number (2015~2020, total = {})'.format(df_model['Count'].sum()), fontsize=20)
    ax.legend(patches, labels, loc='best', bbox_to_anchor=(1.1, 1), fontsize=18)
    #plt.savefig('figures/' + ProgramType + '_IPG_found_in_data_model_pie.png', bbox_inches='tight')

### Longevity & Replacement Analysis

In [ ]:
df_hy = df.groupby('Implant Time').agg({'Serial Number (hashed)':'count'}).reset_index()
df_hy.columns = ['Implant Time', 'Partial Count']
df_implant = pd.merge(dfos_hy, df_hy, left_on=['Time in Half Year'], right_on=['Implant Time'], how='inner').drop('Implant Time', axis=1).sort_values('Time in Half Year').fillna(0).reset_index(drop=True)[['Time in Half Year', 'US Total', 'Partial Count']]
df_implant.columns = ['Implant Time', 'US Total', 'Partial Count']
fig,ax = plt.subplots()
fig.set_size_inches(20,12)
#plt.grid(axis='y')
ax.set_axisbelow(True)
ax.yaxis.grid(color='gray', linestyle='dashed')

vals = [df_implant['Partial Count'],df_implant['US Total']]
X = df_implant['Implant Time']
n = len(vals)
_X = np.arange(len(X))
width = 0.8
for i, color in zip(range(n), ['blue', 'gray']):
    ax.bar(_X - width/2. + i/float(n)*width, vals[i], width=width/float(n), align="edge", color=color)
for i in range(len(vals[0])):
    ax.text(x = _X[i] - width/2 - width/20, y = vals[0][i] + max(vals[0])*0.03, s = '{}'.format(vals[0][i]), size=18)
for i in range(len(vals[1])):
    ax.text(x = _X[i] - width/20, y = vals[1][i] + max(vals[1])*0.03, s = '{}'.format(vals[1][i]), size=18)
for i in range(len(vals[0])):
    ax.text(x = _X[i] - width/2 - width/20, y = vals[0][i] + max(vals[1])*0.06, s = '{:.1f}%'.format(vals[0][i]/vals[1][i]*100), size=20, color='red', fontweight='bold')

plt.xticks(_X, X)
ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.yaxis.set_ticks(np.arange(start, end, stepsize))
ax.set_xlabel('Time of Sales (in Half Year)', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)
ax.set_title(('Percentage of Actual IPG Sales with Programmer Data ' + ProgramType), fontsize=20, fontweight='bold')
ax.legend(['Number of Implants based on Collected Programmer Data','Number of Implants based on Sales Data'],
          loc='best', fontsize=20)
plt.tick_params(labelright=True)
#plt.savefig('figures/' + ProgramType + '_IPG_implant_date.png')

In [ ]:
df_qt = df.groupby('Implant Time Quarter').agg({'Serial Number (hashed)':'count'}).reset_index()
df_qt.columns = ['Implant Time Quarter', 'Partial Count']
df_implant_quarter = pd.merge(dfos_qt, df_qt, left_on=['Time in Quarter'], right_on=['Implant Time Quarter'], how='inner').drop('Implant Time Quarter', axis=1).sort_values('Time in Quarter').fillna(0).reset_index(drop=True)[['Time in Quarter', 'US Total', 'Partial Count']]
df_implant_quarter.columns = ['Implant Time Quarter', 'US Total', 'Partial Count']
fig,ax = plt.subplots()
fig.set_size_inches(20,12)
#plt.grid(axis='y')
ax.set_axisbelow(True)
ax.yaxis.grid(color='gray', linestyle='dashed')

vals = [df_implant_quarter['Partial Count'],df_implant_quarter['US Total']]
X = df_implant_quarter['Implant Time Quarter']
n = len(vals)
_X = np.arange(len(X))
width = 0.8
for i, color in zip(range(n), ['blue', 'gray']):
    ax.bar(_X - width/2. + i/float(n)*width, vals[i], width=width/float(n), align="edge", color=color)
for i in range(len(vals[0])):
    ax.text(x = _X[i] - width/2 - width/10, y = vals[0][i] + max(vals[0])*0.03, s = '{}'.format(vals[0][i]), size=16)
for i in range(len(vals[1])):
    ax.text(x = _X[i] - width/10, y = vals[1][i] + max(vals[1])*0.03, s = '{}'.format(vals[1][i]), size=16)
for i in range(len(vals[0])):
    ax.text(x = _X[i] - width/2 - width/10, y = vals[0][i] + max(vals[1])*0.06, s = '{:.0f}%'.format(vals[0][i]/vals[1][i]*100), size=16, color='red', fontweight='bold')

plt.xticks(_X, X)
ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12) 
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.yaxis.set_ticks(np.arange(start, end, stepsize))
ax.set_xlabel('Time of Sales (in Quarter)', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)
ax.set_title(('Percentage of Actual IPG Sales with Programmer Data ' + ProgramType), fontsize=20, fontweight='bold')
ax.legend(['Number of Implants based on Collected Programmer Data','Number of Implants based on Sales Data'],
          loc='best', fontsize=20)
# for tick in ax.get_xticklabels():
#     tick.set_rotation(90)
plt.xticks(rotation='vertical')
plt.tick_params(labelright=True)
plt.savefig('figures/' + ProgramType + '_IPG_implant_date.png')

In [ ]:
from matplotlib.ticker import PercentFormatter
fig,ax = plt.subplots()
fig.set_size_inches(20,10)
#ax2 = ax.twinx()
ylim_longevity = []
for longevity_col, estimate, impedance, imp_est, color, hist_kws in zip(
    ['Longevity_low', 'Longevity_medium', 'Longevity_high'],
    ['Lowest', 'Medium', 'Highest'],           
    ['highest impedance', 'default impedance', 'lowest impedance'],
    [impedance_high, impedance_default, impedance_low],
    ['blue', 'green', 'gray'],
    [dict(edgecolor='blue', linewidth=3, ls='dotted', fill=False, alpha=1), dict(color='green', fill=True, alpha=0.8),
     dict(edgecolor='gray', linewidth=3, fill=False)]):

#     fig,ax = plt.subplots()
#     fig.set_size_inches(20,10)

    #sns.distplot(df[longevity_col], bins=14, kde=False, color=color, ax=ax, hist_kws=hist_kws)
    
    ax.hist(df[longevity_col], bins=14, histtype='bar', **hist_kws)
    ax.grid(axis='y')
    
    # display percentage on y axis trick 1
    #hist, bins = np.histogram(df[longevity_col], bins=14)
    #ax.bar(bins[:-1], hist.astype(np.float32) / hist.sum(), width=(bins[1]-bins[0]), **hist_kws)
    
    # display percentage on y axis trick 2
    #plt.hist(df[longevity_col], weights=np.ones(len(df[longevity_col])) / len(df[longevity_col]), bins=14, **hist_kws)
    #plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
    
    #ax2.set_ylabel('Percentage (%)', color='red')  # we already handled the x-label with ax1
    #ax2.tick_params(axis='y', labelcolor='red')
    
    plt.xticks(np.arange(0, 17, step=1)) #plt.xticks(fontsize=14, rotation=90)
    ax.tick_params(axis='both', which='major', labelsize=18)
    ax.tick_params(axis='both', which='minor', labelsize=12)
    
    if not ylim_longevity:
        _, ylim_longevity = ax.get_ylim()
        ylim_longevity = ylim_longevity * 1.5
        
    ax.set(ylim=[0, ylim_longevity])
    start, end = ax.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax.yaxis.set_ticks(np.arange(start, end, stepsize))
    
    ax.set_xlabel('Longevity (Years after implant date)', fontsize=20)
    ax.set_ylabel('IPG Counts', fontsize=20)
    ax.set_title('Longevity Distribution of {} Orion IPGs found in Data '.format(len(df)) + ProgramType, fontsize=20, fontweight='bold')
    
#     ax2.hist(df[longevity_col], weights=np.ones(len(df[longevity_col])) / len(df[longevity_col]), 
#              bins=14, histtype='bar', **hist_kws)
#     plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
#     ax2.tick_params(axis='y', labelcolor='red', labelsize=18)
#     ax2.set_ylabel('Percentage of Total', fontsize=20, color='red')
    
#     ax2.set(ylim=[0, ylim_longevity/len(df[longevity_col])])
#     start, end = ax2.get_ylim()
#     ax2.yaxis.set_ticks(np.arange(start, end, 0.01))
    
    #ax.get_legend().remove()
    #plt.legend(['Average = {:.1f} (Years)'.format(df[longevity_col].mean())], loc='best', fontsize=20)
    #plt.savefig('figures/' + ProgramType + '_Longevity_' + estimate + '_estimate.png')
plt.tick_params(labelright=True)
plt.legend(['Lowest estimate, Average = {:.1f} (Years)'.format(df['Longevity_low'].mean()) + '\nbased on highest impedance = {}'.format(impedance_high),
            r'$\bf{' + f"Medium\ estimate,\ Average\ =\ {df['Longevity_medium'].mean():.1f}\ (Years)" + '}$'+ '\nbased on default impedance = {}'.format(impedance_default),
           'Highest estimate, Average = {:.1f} (Years)'.format(df['Longevity_high'].mean()) + '\nbased on lowest impedance = {}'.format(impedance_low)],
           loc='center', bbox_to_anchor=(0.5, -0.3), fontsize=20)
plt.savefig('figures/' + ProgramType + '_Longevity_estimate.png', bbox_inches='tight')

In [ ]:
df_810 = df[['Longevity_low', 'Longevity_medium', 'Longevity_high']].apply(stats.hmean, axis=1)
print(ProgramType,'longevity between 8 and 10 years:',len(df[(df_810>=8) & (df_810<=10)]),'over',len(df))

In [ ]:
dfos_r_hy = dfos_r.groupby(['Time in Half Year']).agg({'Invoiced Units':'sum'}).reset_index()
dfos_r_hy.columns = ['Time in Half Year', 'US Total']
#dfos_r_hy['US Total'] = dfos_r_hy['US Total'].astype(np.int64)

In [ ]:
dfos_r_qt = dfos_r.groupby(['Time in Quarter']).agg({'Invoiced Units':'sum'}).reset_index()
dfos_r_qt.columns = ['Time in Quarter', 'US Total']
#dfos_r_qt['US Total'] = dfos_r_qt['US Total'].astype(np.int64)

In [ ]:
dfos_r_hy = dfos_r_hy[:-1]
dfos_r_qt = dfos_r_qt[:-1]

In [ ]:
dfos_r_hy = dfos_r_hy[:-1]
dfos_r_qt = dfos_r_qt[:-2]

## Results in Half Year

In [ ]:
df['Replacement Date Earliest'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_low']-avg_days_adjust), axis=1)
df['Replacement Date Average'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_medium']-avg_days_adjust), axis=1)
df['Replacement Date Latest'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_high']-avg_days_adjust), axis=1)

df['Replacement Earliest'] = df['Replacement Date Earliest'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
df['Replacement Average'] = df['Replacement Date Average'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
df['Replacement Latest'] = df['Replacement Date Latest'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
df['Replacement Earliest Quarter'] = df['Replacement Date Earliest'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
df['Replacement Average Quarter'] = df['Replacement Date Average'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
df['Replacement Latest Quarter'] = df['Replacement Date Latest'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))

In [ ]:
display_hy_range = [(str(Y) + 'H' + '{}'.format(h)) for Y in display_year_range0 for h in range(1, 3)]

In [ ]:
for replacement_col, estimate, impedance, imp_est in zip(['Earliest', 'Average', 'Latest'], ['Earliest', 'Medium', 'Latest'], 
                                                         ['highest impedance', 'default impedance', 'lowest impedance'],
                                                         [impedance_high, impedance_default, impedance_low]):

    df_hy = df.groupby(['Implant Time', 'Replacement ' + replacement_col]).agg({'Serial Number (hashed)':'count'}).reset_index()
    df_hy.columns = ['Implant Time', 'Replacement ' + replacement_col, 'Replacement Count Raw']
    total = df_hy['Replacement Count Raw'].sum()
    
    df_reshape = df_hy.pivot(index='Replacement ' + replacement_col, columns='Implant Time', values='Replacement Count Raw').fillna(0)#.astype(int)
    #df_reshape = df_reshape[df_reshape.sum(axis=1) > 0.0001*total]
    df_reshape = df_reshape[df_reshape.index.isin([(str(Y) + 'H' + '{}'.format(h)) for Y in display_year_range for h in range(1, 3)])]
    exec('df_reshape_hy_'+estimate+'_noforecast=df_reshape.sum(axis=1)')
    
    colors = zip([i for i in np.linspace(0, 1, len(dfos_hy))], [0.3]*len(dfos_hy), [0.8]*len(dfos_hy))

    fig,ax = plt.subplots()
    df_reshape.plot(kind='bar', stacked=True, ax=ax, figsize=(20,12), fontsize=18) #, color=colors)
    ax.grid(axis='y')
    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='both', which='minor', labelsize=16)
    #ax.set(ylim=[0, ylim_replacement])
    start, end = ax.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax.set_xlabel('Expected Replacement Date (in Half Year)', fontsize=20)
    ax.set_ylabel('IPG Counts', fontsize=20)
    ax.set_title(r'$\bf{' + f"Replacement\ Timeline\ of\ {total}\ IPGs " + ProgramType + '}$' +
                 '\n-- ' + 
                 r'$\bf{' + estimate + '}$' +
                 ' expectation based on ' + impedance + ' = ' + imp_est, fontsize=20)
    ax.legend('Sales in ' + df_reshape.columns, loc='upper right', bbox_to_anchor=(1.4, 1), fontsize=20);
        
#     ax2 = ax.twinx()
#     (df_reshape/total).plot(kind='bar', stacked=True, ax=ax2, figsize=(20,12), fontsize=18, color=colors)
#     #df_reshape.plot.bar(stacked=True, ax=ax2, figsize=(20,12), fontsize=18)
#     plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
    
#     ax2.tick_params(axis='y', labelcolor='red', labelsize=20)
#     #ax2.set(ylim=[0, ylim_replacement/total])
#     start, end = ax2.get_ylim()
#     ax2.yaxis.set_ticks(np.arange(start, end, 0.01))
#     ax2.set_ylabel('Percentage of Total', fontsize=20, color='red')

#     ax.get_legend().remove()
#     ax2.legend('Sales in ' + df_reshape.columns, loc='upper right', bbox_to_anchor=(1.4, 1), fontsize=20);
    
    plt.tick_params(labelright=True)
    #plt.savefig('figures/' + ProgramType + '_replacement_' + estimate + '_estimate_hy.png', bbox_inches='tight')

In [ ]:
for replacement_col, estimate, impedance, imp_est in zip(['Earliest', 'Average', 'Latest'], ['Earliest', 'Medium', 'Latest'], 
                                                         ['highest impedance', 'default impedance', 'lowest impedance'],
                                                         [impedance_high, impedance_default, impedance_low]):

    df_hy = df.groupby(['Implant Time', 'Replacement ' + replacement_col]).agg({'Serial Number (hashed)':'count'}).reset_index()
    df_hy.columns = ['Implant Time', 'Replacement ' + replacement_col, 'Replacement Count Raw']
    total = df_implant['US Total'].sum()
    
    df_adj = pd.merge(df_hy, df_implant, on='Implant Time', how='left')
    df_adj['Replacement Count Adjusted US'] = (df_adj['Replacement Count Raw'] * df_adj['US Total']/df_adj['Partial Count'])
    df_adj['Replacement Count Adjusted US'] = df_adj['Replacement Count Adjusted US'].replace([np.inf, -np.inf, np.nan], 0).round().astype(int)
    
    df_reshape = df_adj.pivot(index='Replacement ' + replacement_col, columns='Implant Time', values='Replacement Count Adjusted US').fillna(0)#.astype(int)
    #df_reshape = df_reshape[df_reshape.sum(axis=1) > 0.0001*total]
    df_reshape = df_reshape[df_reshape.index.isin([(str(Y) + 'H' + '{}'.format(h)) for Y in display_year_range for h in range(1, 3)])]
    exec('df_reshape_hy_'+estimate+'_noforecast=df_reshape.sum(axis=1)')

    colors = zip([i for i in np.linspace(0, 1, len(dfos_hy))], [0.3]*len(dfos_hy), [0.8]*len(dfos_hy))

    fig,ax = plt.subplots()
    df_reshape.plot(kind='bar', stacked=True, ax=ax, figsize=(20,12), fontsize=18) #, color=colors)
    ax.grid(axis='y')
    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='both', which='minor', labelsize=16)
    #ax.set(ylim=[0, ylim_replacement])
    start, end = ax.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax.set_xlabel('Expected Replacement Date (in Half Year)', fontsize=20)
    ax.set_ylabel('IPG Counts', fontsize=20)
    ax.set_title(r'$\bf{' + f"Replacement\ Timeline\ of\ {total}\ IPGs " + ProgramType + '}$' +
                 ', Adjusted by US Total Sales' + '\n-- ' + 
                 r'$\bf{' + estimate + '}$' +
                 ' expectation based on ' + impedance + ' = ' + imp_est, fontsize=20)
    ax.legend('Sales in ' + df_reshape.columns, loc='upper right', bbox_to_anchor=(1.4, 1), fontsize=20);
                
#     ax2 = ax.twinx()
#     (df_reshape/total).plot(kind='bar', stacked=True, ax=ax2, figsize=(20,12), fontsize=18, color=colors)
#     #df_reshape.plot.bar(stacked=True, ax=ax2, figsize=(20,12), fontsize=18)
#     plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
    
#     ax2.tick_params(axis='y', labelcolor='red', labelsize=20)
#     #ax2.set(ylim=[0, ylim_replacement/total])
#     start, end = ax2.get_ylim()
#     ax2.yaxis.set_ticks(np.arange(start, end, 0.01))
#     ax2.set_ylabel('Percentage of Total', fontsize=20, color='red')

#     ax.get_legend().remove()
#     ax2.legend('Sales in ' + df_reshape.columns, loc='upper right', bbox_to_anchor=(1.4, 1), fontsize=20);
    
    plt.tick_params(labelright=True)
    #plt.savefig('figures/' + ProgramType + '_replacement_' + estimate + '_estimate_hy.png', bbox_inches='tight')

In [ ]:
random_seed=1234567
df_predict_low = []
df_ = df[['Serial Number (hashed)', 'Longevity_low', 'Longevity_medium', 'Longevity_high']]
for pred_date, pred_count, pred_date1 in zip(dfos_predict_hy_low['Time in Half Year'], dfos_predict_hy_low['US Total'],
    [(str(Y) + '{}'.format(h)) for Y in predict_year_range for h in ['-04-01', '-10-01']][1:]):

    df_tmp = df_.sample(n=pred_count, replace=True, random_state=random_seed+pred_count)
    df_tmp['Serial Number (hashed)'] = 'Predicted'

    df_tmp['Implant Date'] = pd.to_datetime(pred_date1, format='%Y-%m-%d')
    df_tmp['Replacement Date Earliest'] = df_tmp.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_low']-avg_days_adjust), axis=1)
    df_tmp['Replacement Date Average'] = df_tmp.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_medium']-avg_days_adjust), axis=1)
    df_tmp['Replacement Date Latest'] = df_tmp.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_high']-avg_days_adjust), axis=1)
    df_tmp['Implant Time'] = pred_date
    df_tmp['Replacement Earliest'] = df_tmp['Replacement Date Earliest'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
    df_tmp['Replacement Average'] = df_tmp['Replacement Date Average'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
    df_tmp['Replacement Latest'] = df_tmp['Replacement Date Latest'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
    df_predict_low.append(df_tmp)
    
df_predict_low = pd.concat(df_predict_low, axis=0, sort=True).reset_index(drop=True)

# df_predict_high = []
# for pred_date, pred_count, pred_date1 in zip(dfos_predict_hy_high['Time in Half Year'], dfos_predict_hy_high['US Total'],
#     [(str(Y) + '{}'.format(h)) for Y in predict_year_range for h in ['-04-01', '-10-01']]):

#     df_tmp = df_.sample(n=pred_count, replace=True, random_state=random_seed+pred_count)
#     df_tmp['Serial Number (hashed)'] = 'Predicted'

#     df_tmp['Implant Date'] = pd.to_datetime(pred_date1, format='%Y-%m-%d')
#     df_tmp['Replacement Date Earliest'] = df_tmp.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_low']-avg_days_adjust), axis=1)
#     df_tmp['Replacement Date Average'] = df_tmp.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_medium']-avg_days_adjust), axis=1)
#     df_tmp['Replacement Date Latest'] = df_tmp.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_high']-avg_days_adjust), axis=1)
#     df_tmp['Implant Time'] = pred_date
#     df_tmp['Replacement Earliest'] = df_tmp['Replacement Date Earliest'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
#     df_tmp['Replacement Average'] = df_tmp['Replacement Date Average'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
#     df_tmp['Replacement Latest'] = df_tmp['Replacement Date Latest'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
#     df_predict_high.append(df_tmp)
    
# df_predict_high = pd.concat(df_predict_high, axis=0, sort=True).reset_index(drop=True)

In [ ]:
fig_rev,ax_rev = plt.subplots()
fig_rev.set_size_inches(16,11)
ax_rev2 = ax_rev.twinx()

fig_trev,ax_trev = plt.subplots()
fig_trev.set_size_inches(16,11)
ax_trev2 = ax_trev.twinx()
############ low estimate for predicted sales
for replacement_col, estimate, impedance, imp_est, ls, tls in zip(['Earliest', 'Average', 'Latest'],
                                                 ['Earliest', 'Medium', 'Latest'], 
                                                 ['highest impedance', 'default impedance', 'lowest impedance'],
                                                 [impedance_high, impedance_default, impedance_low],
                                                 [dict(linewidth=2, ls='dotted', color='gray', marker='.', markersize=10),
                                                  dict(linewidth=3, color='blue', marker='.', markersize=20),
                                                  dict(linewidth=2, ls='dashed', color='gray', marker='.', markersize=10)],
                                                 [dict(linewidth=4, ls='dotted', color='gray', marker='*', markersize=10),
                                                  dict(linewidth=5, color='blue', marker='*', markersize=20),
                                                  dict(linewidth=4, ls='dashed', color='gray', marker='*', markersize=10)]):

    df_hy = df.groupby(['Implant Time', 'Replacement ' + replacement_col]).agg({'Serial Number (hashed)':'count'}).reset_index()
    df_hy.columns = ['Implant Time', 'Replacement ' + replacement_col, 'Replacement Count Raw']

    df_hy_predict = df_predict_low.groupby(['Implant Time', 'Replacement ' + replacement_col]).agg({'Serial Number (hashed)':'count'}).reset_index()
    df_hy_predict.columns = ['Implant Time', 'Replacement ' + replacement_col, 'Replacement Count Raw']
    total = df_implant['US Total'].sum() + df_hy_predict['Replacement Count Raw'].sum()

    df_adj = pd.merge(df_hy, df_implant, on='Implant Time', how='left')
    df_adj['Replacement Count Adjusted US'] = (df_adj['Replacement Count Raw'] * df_adj['US Total']/df_adj['Partial Count'])
    df_adj['Replacement Count Adjusted US'] = df_adj['Replacement Count Adjusted US'].replace([np.inf, -np.inf, np.nan], 0).round().astype(int)

    df_adj_predict = pd.concat([df_adj, df_hy_predict], axis=0, sort=True).reset_index(drop=True)
    df_adj_predict['Replacement Count Adjusted US'].fillna(df_adj_predict['Replacement Count Raw'], inplace=True)    

    df_reshape_low = df_adj_predict.pivot(index='Replacement ' + replacement_col, columns='Implant Time', values='Replacement Count Adjusted US').fillna(0)#.astype(int)
    #df_reshape_low = df_reshape_low[df_reshape_low.sum(axis=1) > 0.0001*total]
    df_reshape_low = df_reshape_low[df_reshape_low.index.isin([(str(Y) + 'H' + '{}'.format(h)) for Y in display_year_range for h in range(1, 3)])]
    exec('df_reshape_hy_'+estimate+'=df_reshape_low.sum(axis=1)')

    colors = zip([i for i in np.linspace(0, 1, len(dfos_hy))] + [i for i in np.linspace(0, 1, len(dfos_predict_hy_low))],
                 [0.3]*len(dfos_hy) + [0.8]*len(dfos_predict_hy_low), [0.8]*len(dfos_hy) + [0]*len(dfos_predict_hy_low))
    
    fig,ax = plt.subplots()
    df_reshape_low.plot(kind='bar', stacked=True, ax=ax, figsize=(20,12), fontsize=20, color=colors)
    ax.grid(axis='y')
    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='both', which='minor', labelsize=16)  
#     if not ylim_replacement:
#         _, ylim_replacement = ax.get_ylim()
#         ylim_replacement = ylim_replacement * 1.3
    #ax.set(ylim=[0, ylim_replacement])
    start, end = ax.get_ylim()
    stepsize = Stepsize_Cal(end) 
    ax.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax.set_xlabel('Expected Replacement Date (in Half Year)', fontsize=20)
    ax.set_ylabel('IPG Counts', fontsize=20)
    ax.legend(list('Sales in ' + df_reshape_low.columns[:len(dfos_hy)]) + list('Predicted New Sales in ' + df_reshape_low.columns[-len(dfos_predict_hy_low):]),
              loc='upper right', bbox_to_anchor=(1.45, 1), fontsize=20)
    
#     ax2 = ax.twinx()
#     (df_reshape_low/total).plot(kind='bar', stacked=True, ax=ax2, figsize=(20,12), fontsize=18, color=colors)
#     #df_reshape_low.plot.bar(stacked=True, ax=ax2, figsize=(20,12), fontsize=18)
#     plt.gca().yaxis.set_major_formatter(PercentFormatter(1))

#     ax2.tick_params(axis='y', labelcolor='red', labelsize=20)
#     #ax2.set(ylim=[0, ylim_replacement/total])
#     start, end = ax2.get_ylim()
#     ax2.yaxis.set_ticks(np.arange(start, end, 0.01))
#     ax2.set_ylabel('Percentage of Total', fontsize=20, color='red')

#     ax.get_legend().remove()
#     ax2.legend(list('Sales in ' + df_reshape_low.columns[:len(dfos_hy)]) + list('Predicted New Sales in ' + df_reshape_low.columns[-len(dfos_predict_hy_low):]),
#                loc='upper right', bbox_to_anchor=(1.45, 1), fontsize=20)

    ax.set_title(r'$\bf{' + f"Replacement\ Timeline\ of\ {total}\ IPGs " + ProgramType + '}$' +
                 ' Based on Low Estimation of Predicted New Sales' + '\n-- ' + 
                 r'$\bf{' + estimate + '}$' +
                 ' expectation based on ' + impedance + ' = ' + imp_est, fontsize=20)
    plt.tick_params(labelright=True)
    #fig.savefig('figures/' + ProgramType + '_replacement_' + estimate + '_estimate_predict_low.png', bbox_inches='tight')
    
    ######### replacement count
    ax_rev.plot(df_reshape_low.index[df_reshape_low.index.isin(display_hy_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_hy_range)], **ls)
    ax_rev.tick_params(axis='both', which='major', labelsize=20)
    ax_rev.tick_params(axis='both', which='minor', labelsize=16)
    #ax_rev.set(ylim=[0, ylim_replacement])
    start, end = ax_rev.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_rev.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_rev.set_xlabel('Time of Sales (in Half Year)', fontsize=20)
    ax_rev.set_ylabel('IPG Counts', fontsize=20)
    
    ax_rev2.plot(df_reshape_low.index[df_reshape_low.index.isin(display_hy_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_hy_range)], **ls)
    ax_rev2.tick_params(axis='y', labelsize=20)
    #ax_rev2.set(ylim=[0, ylim_replacement]); 
    start, end = ax_rev2.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_rev2.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_rev2.set_ylabel('IPG Counts', fontsize=20)

    ######### total count
    if estimate == 'Earliest':
        dfos_hy_low = pd.concat([dfos_hy, dfos_predict_hy_low], axis=0, sort=True).sort_values(['Time in Half Year']).reset_index(drop=True)[['Time in Half Year', 'US Total']]
        dfos_hy_low = dfos_hy_low[dfos_hy_low['Time in Half Year'].isin(df_reshape_low.index[df_reshape_low.index.isin(display_hy_range)].tolist())]
        #ax_trev.plot(dfos_hy_low['Time in Half Year'], dfos_hy_low['US Total'], **dict(ls='dashed', linewidth=3, color='blue', marker='.', markersize=20))

    dfos_total_hy_low = pd.Series(dfos_hy_low['US Total'].tolist(), index=dfos_hy_low['Time in Half Year']).add(
        df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_hy_range)], fill_value=0)
    ax_trev.plot(dfos_total_hy_low.index, dfos_total_hy_low, **tls)
    ax_trev.plot(df_reshape_low.index[df_reshape_low.index.isin(display_hy_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_hy_range)], **ls)
    ax_trev.tick_params(axis='both', which='major', labelsize=20)
    ax_trev.tick_params(axis='both', which='minor', labelsize=16)
    #ax_trev.set(ylim=[0, ylim_replacement])
    start, end = ax_trev.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_trev.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_trev.set_xlabel('Time of Sales (in Half Year)', fontsize=20)
    ax_trev.set_ylabel('IPG Counts', fontsize=20)
    
    ax_trev2.plot(dfos_total_hy_low.index, dfos_total_hy_low, **tls)
    ax_trev2.plot(df_reshape_low.index[df_reshape_low.index.isin(display_hy_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_hy_range)], **ls)
    ax_trev2.tick_params(axis='y', labelsize=20)
    #ax_trev2.set(ylim=[0, ylim_replacement]); 
    start, end = ax_trev2.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_trev2.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_trev2.set_ylabel('IPG Counts', fontsize=20)

ax_rev.grid(axis='y')
lgd = [(fitting + '\n' + estimate + ' expectation of replacement') 
       for fitting in ['Based on low estimation of predicted new sales']
       for estimate in ['& earliest', '& medium', '& latest']]
ax_rev.legend(lgd, loc='center', bbox_to_anchor=(1.4, 0.5), fontsize=16, ncol=1, handleheight=0)    
ax_rev.set_title(('US Orion IPG Replacement Count Prediction ' + ProgramType), fontsize=20, fontweight='bold')
# locs, labels = plt.xticks()
# ax_rev.set_xticklabels(labels, rotation=90)
for tick in ax_rev.get_xticklabels():
    tick.set_rotation(90)
#fig_rev.savefig('figures/' + ProgramType + '_replacement_count_predict_hy.png', bbox_inches='tight')

ax_trev.grid(axis='y')
lgd = [(rev + '\n' + fitting + '\n' + estimate + ' expectation of replacement)')
       for fitting in ['(based on low estimation of predicted new sales']
       for estimate in ['& earliest', '& medium', '& latest']
       for rev in ['New Sales + Replacement', 'Replacement Only']]
ax_trev.legend(lgd, loc='center', bbox_to_anchor=(1.4, 0.42), fontsize=16, ncol=1, handleheight=0)    
ax_trev.set_title(('US Orion Total Count Prediction (New Sales + Replacement) ' + ProgramType), fontsize=20, fontweight='bold')
# locs, labels = plt.xticks()
# ax_trev.set_xticklabels(labels, rotation=90)
for tick in ax_trev.get_xticklabels():
    tick.set_rotation(90)
#fig_trev.savefig('figures/' + ProgramType + '_total_count_predict_hy.png', bbox_inches='tight')

In [ ]:
df_replacement_hy = pd.DataFrame({'Earliest': df_reshape_hy_Earliest,
                                  'Medium': df_reshape_hy_Medium,
                                  'Latest': df_reshape_hy_Latest})
df_replacement_hy.fillna(0, inplace=True)
df_replacement_hy.sort_index(inplace=True)

In [ ]:
df_replacement_hy_noforecast = pd.DataFrame({'Earliest': df_reshape_hy_Earliest_noforecast,
                                             'Medium': df_reshape_hy_Medium_noforecast,
                                             'Latest': df_reshape_hy_Latest_noforecast})
df_replacement_hy_noforecast.fillna(0, inplace=True)
df_replacement_hy_noforecast.sort_index(inplace=True)

In [ ]:
replacement_predict_hy = df_replacement_hy.transpose().describe().transpose().sort_index()['mean']
replacement_predict_std_hy = df_replacement_hy.transpose().describe().transpose().sort_index()['std']

replacement_predict_hy_noforecast = df_replacement_hy_noforecast.transpose().describe().transpose().sort_index()['mean']
replacement_predict_std_hy_noforecast = df_replacement_hy_noforecast.transpose().describe().transpose().sort_index()['std']

In [ ]:
replacement_predict_hy = pd.Series(0, index=display_hy_range).add(replacement_predict_hy, fill_value=0)
replacement_predict_std_hy = pd.Series(0, index=display_hy_range).add(replacement_predict_std_hy, fill_value=0)

replacement_predict_hy_noforecast = pd.Series(0, index=display_hy_range).add(replacement_predict_hy_noforecast, fill_value=0)
replacement_predict_std_hy_noforecast = pd.Series(0, index=display_hy_range).add(replacement_predict_std_hy_noforecast, fill_value=0)

In [ ]:
# without sales forecast
fig,ax = plt.subplots()
fig.set_size_inches(20,10)
ax2 = ax.twinx()
ax.grid(axis='y')
# ax.set_axisbelow(True)
# ax.yaxis.grid(color='gray', linestyle='dashed')

display_skip = 0

ax.plot(replacement_predict_hy_noforecast.index[replacement_predict_hy_noforecast.index.isin(display_hy_range)], 
        replacement_predict_hy_noforecast.values[replacement_predict_hy_noforecast.index.isin(display_hy_range)],
        linewidth=3, marker='.', markersize=20, color='black')
ax.plot(df_replacement_hy_noforecast['Earliest'].index[df_replacement_hy_noforecast['Earliest'].index.isin(display_hy_range)],
        df_replacement_hy_noforecast['Earliest'].values[df_replacement_hy_noforecast['Earliest'].index.isin(display_hy_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='blue')
ax.plot(df_replacement_hy_noforecast['Medium'].index[df_replacement_hy_noforecast['Medium'].index.isin(display_hy_range)],
        df_replacement_hy_noforecast['Medium'].values[df_replacement_hy_noforecast['Medium'].index.isin(display_hy_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='green')
ax.plot(df_replacement_hy_noforecast['Latest'].index[df_replacement_hy_noforecast['Latest'].index.isin(display_hy_range)],
        df_replacement_hy_noforecast['Latest'].values[df_replacement_hy_noforecast['Latest'].index.isin(display_hy_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='purple')
ax.plot(dfos_r_hy['Time in Half Year'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)][display_skip:],
        dfos_r_hy['US Total'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)][display_skip:],
        linestyle='dotted', marker='.', markersize=10, color='red', alpha=0.6)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time in Half Year', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

ax2.set(ylim=[0, end]); 
ax2.tick_params(axis='y', labelsize=18)
ax2.yaxis.set_ticks(np.arange(0, end, stepsize))
ax2.set_ylabel('Orion IPG Counts', fontsize=20)

for tick in ax.get_xticklabels():
    tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life (average of a, b & c)',
                 '  a) Predicted Replacement (earliest, based on highest impedance)',
                 '  b) Predicted Replacement (medium, based on medium impedance)',
                 '  c) Predicted Replacement (latest, based on lowest impedance)',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)'],
                loc='best', fontsize=20)
#plt.savefig('figures/' + ProgramType + '_IPG_replacement_hy_noforecast.png')

In [ ]:
# without sales forecast
fig,ax = plt.subplots()
fig.set_size_inches(20,12)
ax2 = ax.twinx()
ax.grid(axis='y')
# ax.set_axisbelow(True)
# ax.yaxis.grid(color='gray', linestyle='dashed')

display_skip = 0 

ax.plot(replacement_predict_hy_noforecast.index[replacement_predict_hy_noforecast.index.isin(display_hy_range)], 
        replacement_predict_hy_noforecast.values[replacement_predict_hy_noforecast.index.isin(display_hy_range)],
        linewidth=3, marker='.', markersize=20, color='black')
ax.plot(dfos_r_hy['Time in Half Year'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)][display_skip:],
        dfos_r_hy['US Total'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)][display_skip:],
        linestyle='dotted', marker='.', markersize=10, color='red', alpha=0.6)
ax.errorbar(replacement_predict_hy_noforecast.index[replacement_predict_hy_noforecast.index.isin(display_hy_range)], 
            replacement_predict_hy_noforecast.values[replacement_predict_hy_noforecast.index.isin(display_hy_range)],
            yerr = replacement_predict_std_hy_noforecast.values[replacement_predict_std_hy_noforecast.index.isin(display_hy_range)],
            linewidth=2, color='gray', barsabove=True, capsize=5)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time in Half Year', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

ax2.set(ylim=[0, end]); 
ax2.tick_params(axis='y', labelsize=18)
ax2.yaxis.set_ticks(np.arange(0, end, stepsize))
ax2.set_ylabel('Orion IPG Counts', fontsize=20)

for tick in ax.get_xticklabels():
    tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)',
                 'Range of Predicted Replacement'],
                loc='best', fontsize=20)
#plt.savefig('figures/' + ProgramType + '_IPG_replacement_hy_std_noforecast.png')

In [ ]:
# without sales forecast
fig,ax = plt.subplots()
fig.set_size_inches(20,10)
ax.grid(axis='y')
ax.set_axisbelow(True)
ax.yaxis.grid(color='gray', linestyle='dashed')

width = 0.8

bars1 = replacement_predict_hy_noforecast.values[replacement_predict_hy_noforecast.index.isin(display_hy_range)]
std1 = replacement_predict_std_hy_noforecast.values[replacement_predict_std_hy_noforecast.index.isin(display_hy_range)]
bars2 = np.array(dfos_r_hy['US Total'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)])
X1 = replacement_predict_hy_noforecast.index[replacement_predict_hy_noforecast.index.isin(display_hy_range)]
X2 = dfos_r_hy['Time in Half Year'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)]
_X1 = np.arange(len(X1))
_X2 = np.arange(len(X2))
ax.bar(_X1 - width/2, bars1, width=width/2, align="edge", color='black',
       yerr=std1, capsize=5)
ax.bar(_X2, bars2, width=width/2, align="edge", color='red', alpha=0.4)
# for i in range(min(len(X1),len(X2))):
#     ax.text(x = _X1[i] - width/2 - width/20, y = bars1[i] + max(bars1)*0.03, s = '{:.1f}%'.format(bars1[i]/bars2[i]*100), size=20)
for i in range(len(X1)):
    ax.text(x = _X1[i] - width/2, y = bars1[i] + max(bars1)*0.03, s = '{:.0f}'.format(round(bars1[i])), size=20)
for i in range(len(X2)):
    ax.text(x = _X2[i], y = bars2[i] + max(bars2)*0.03, s = '{}'.format(bars2[i]), size=20)
plt.xticks(_X1, X1)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.2
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time in Half Year', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

# for tick in ax.get_xticklabels():
#     tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType +
              '\nPredicted Replacement (due to Battery End of Life) vs. Actual Total Replacements'), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life (bar: prediction range)',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)'],
                loc='best', fontsize=20)
plt.tick_params(labelright=True)
#plt.savefig('figures/' + ProgramType + '_IPG_replacement_predict_hy_noforecast.png')

In [ ]:
fig,ax = plt.subplots()
fig.set_size_inches(20,10)
ax2 = ax.twinx()
ax.grid(axis='y')
# ax.set_axisbelow(True)
# ax.yaxis.grid(color='gray', linestyle='dashed')

display_skip = 0

ax.plot(replacement_predict_hy.index[replacement_predict_hy.index.isin(display_hy_range)], 
        replacement_predict_hy.values[replacement_predict_hy.index.isin(display_hy_range)],
        linewidth=3, marker='.', markersize=20, color='black')
ax.plot(df_replacement_hy['Earliest'].index[df_replacement_hy['Earliest'].index.isin(display_hy_range)],
        df_replacement_hy['Earliest'].values[df_replacement_hy['Earliest'].index.isin(display_hy_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='blue')
ax.plot(df_replacement_hy['Medium'].index[df_replacement_hy['Medium'].index.isin(display_hy_range)],
        df_replacement_hy['Medium'].values[df_replacement_hy['Medium'].index.isin(display_hy_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='green')
ax.plot(df_replacement_hy['Latest'].index[df_replacement_hy['Latest'].index.isin(display_hy_range)],
        df_replacement_hy['Latest'].values[df_replacement_hy['Latest'].index.isin(display_hy_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='purple')
ax.plot(dfos_r_hy['Time in Half Year'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)][display_skip:],
        dfos_r_hy['US Total'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)][display_skip:],
        linestyle='dotted', marker='.', markersize=10, color='red', alpha=0.6)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time in Half Year', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

ax2.set(ylim=[0, end]); 
ax2.tick_params(axis='y', labelsize=18)
ax2.yaxis.set_ticks(np.arange(0, end, stepsize))
ax2.set_ylabel('Orion IPG Counts', fontsize=20)

for tick in ax.get_xticklabels():
    tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life (average of a, b & c)',
                 '  a) Predicted Replacement (earliest, based on highest impedance)',
                 '  b) Predicted Replacement (medium, based on medium impedance)',
                 '  c) Predicted Replacement (latest, based on lowest impedance)',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)'],
                loc='best', fontsize=20)
#plt.savefig('figures/' + ProgramType + '_IPG_replacement_hy.png')

In [ ]:
fig,ax = plt.subplots()
fig.set_size_inches(20,12)
ax2 = ax.twinx()
ax.grid(axis='y')
# ax.set_axisbelow(True)
# ax.yaxis.grid(color='gray', linestyle='dashed')

display_skip = 0

ax.plot(replacement_predict_hy.index[replacement_predict_hy.index.isin(display_hy_range)], 
        replacement_predict_hy.values[replacement_predict_hy.index.isin(display_hy_range)],
        linewidth=3, marker='.', markersize=20, color='black')
ax.plot(dfos_r_hy['Time in Half Year'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)][display_skip:],
        dfos_r_hy['US Total'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)][display_skip:],
        linestyle='dotted', marker='.', markersize=10, color='red', alpha=0.6)
ax.errorbar(replacement_predict_hy.index[replacement_predict_hy.index.isin(display_hy_range)], 
            replacement_predict_hy.values[replacement_predict_hy.index.isin(display_hy_range)],
            yerr = replacement_predict_std_hy.values[replacement_predict_std_hy.index.isin(display_hy_range)],
            linewidth=2, color='gray', barsabove=True, capsize=5)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time in Half Year', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

ax2.set(ylim=[0, end]); 
ax2.tick_params(axis='y', labelsize=18)
ax2.yaxis.set_ticks(np.arange(0, end, stepsize))
ax2.set_ylabel('Orion IPG Counts', fontsize=20)

for tick in ax.get_xticklabels():
    tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)',
                 'Range of Predicted Replacement'],
                loc='best', fontsize=20)
#plt.savefig('figures/' + ProgramType + '_IPG_replacement_hy_std.png')

In [ ]:
fig,ax = plt.subplots()
fig.set_size_inches(20,10)
ax.grid(axis='y')
ax.set_axisbelow(True)
ax.yaxis.grid(color='gray', linestyle='dashed')

width = 0.8

bars1 = replacement_predict_hy.values[replacement_predict_hy.index.isin(display_hy_range)]
std1 = replacement_predict_std_hy.values[replacement_predict_std_hy.index.isin(display_hy_range)]
bars2 = np.array(dfos_r_hy['US Total'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)])
X1 = replacement_predict_hy.index[replacement_predict_hy.index.isin(display_hy_range)]
X2 = dfos_r_hy['Time in Half Year'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)]
_X1 = np.arange(len(X1))
_X2 = np.arange(len(X2))
ax.bar(_X1 - width/2, bars1, width=width/2, align="edge", color='black',
       yerr=std1, capsize=5)
ax.bar(_X2, bars2, width=width/2, align="edge", color='red', alpha=0.4)
# for i in range(min(len(X1),len(X2))):
#     ax.text(x = _X1[i] - width/2 - width/20, y = bars1[i] + max(bars1)*0.03, s = '{:.1f}%'.format(bars1[i]/bars2[i]*100), size=20)
for i in range(len(X1)):
    ax.text(x = _X1[i] - width/2, y = bars1[i] + max(bars1)*0.03, s = '{:.0f}'.format(round(bars1[i])), size=20)
for i in range(len(X2)):
    ax.text(x = _X2[i], y = bars2[i] + max(bars2)*0.03, s = '{}'.format(bars2[i]), size=20)
plt.xticks(_X1, X1)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time in Half Year', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

# for tick in ax.get_xticklabels():
#     tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType +
              '\nPredicted Replacement (due to Battery End of Life) vs. Actual Total Replacements'), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life (bar: prediction range)',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)'],
                loc='best', fontsize=20)
plt.tick_params(labelright=True)
#plt.savefig('figures/' + ProgramType + '_IPG_replacement_predict_hy.png')

### Adjust based on all causes for replacement

In [ ]:
df['Replacement Date Earliest'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=avg_days_replace), axis=1)
df['Replacement Date Average'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=avg_days_replace), axis=1)
df['Replacement Date Latest'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=avg_days_replace), axis=1)

df['Replacement Earliest'] = df['Replacement Date Earliest'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
df['Replacement Average'] = df['Replacement Date Average'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
df['Replacement Latest'] = df['Replacement Date Latest'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))

In [ ]:
df_predict_low['Replacement Date Earliest'] = df_predict_low.apply(lambda x: x['Implant Date'] + timedelta(days=avg_days_replace), axis=1)
df_predict_low['Replacement Date Average'] = df_predict_low.apply(lambda x: x['Implant Date'] + timedelta(days=avg_days_replace), axis=1)
df_predict_low['Replacement Date Latest'] = df_predict_low.apply(lambda x: x['Implant Date'] + timedelta(days=avg_days_replace), axis=1)
df_predict_low['Replacement Earliest'] = df_predict_low['Replacement Date Earliest'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
df_predict_low['Replacement Average'] = df_predict_low['Replacement Date Average'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))
df_predict_low['Replacement Latest'] = df_predict_low['Replacement Date Latest'].apply(lambda x: str(x.year) + 'H' + str(math.ceil(x.month/6)))

In [ ]:
fig_rev,ax_rev = plt.subplots()
fig_rev.set_size_inches(16,11)
ax_rev2 = ax_rev.twinx()

fig_trev,ax_trev = plt.subplots()
fig_trev.set_size_inches(16,11)
ax_trev2 = ax_trev.twinx()
############ low estimate for predicted sales
for replacement_col, estimate, impedance, imp_est, ls, tls in zip(['Earliest', 'Average', 'Latest'],
                                                 ['Earliest', 'Medium', 'Latest'], 
                                                 ['highest impedance', 'default impedance', 'lowest impedance'],
                                                 [impedance_high, impedance_default, impedance_low],
                                                 [dict(linewidth=2, ls='dotted', color='gray', marker='.', markersize=10),
                                                  dict(linewidth=3, color='blue', marker='.', markersize=20),
                                                  dict(linewidth=2, ls='dashed', color='gray', marker='.', markersize=10)],
                                                 [dict(linewidth=4, ls='dotted', color='gray', marker='*', markersize=10),
                                                  dict(linewidth=5, color='blue', marker='*', markersize=20),
                                                  dict(linewidth=4, ls='dashed', color='gray', marker='*', markersize=10)]):

    df_hy = df.groupby(['Implant Time', 'Replacement ' + replacement_col]).agg({'Serial Number (hashed)':'count'}).reset_index()
    df_hy.columns = ['Implant Time', 'Replacement ' + replacement_col, 'Replacement Count Raw']

    df_hy_predict = df_predict_low.groupby(['Implant Time', 'Replacement ' + replacement_col]).agg({'Serial Number (hashed)':'count'}).reset_index()
    df_hy_predict.columns = ['Implant Time', 'Replacement ' + replacement_col, 'Replacement Count Raw']
    total = df_implant['US Total'].sum() + df_hy_predict['Replacement Count Raw'].sum()

    df_adj = pd.merge(df_hy, df_implant, on='Implant Time', how='left')
    df_adj['Replacement Count Adjusted US'] = (df_adj['Replacement Count Raw'] * df_adj['US Total']/df_adj['Partial Count'])
    df_adj['Replacement Count Adjusted US'] = df_adj['Replacement Count Adjusted US'].replace([np.inf, -np.inf, np.nan], 0).round().astype(int)

    df_adj_predict = pd.concat([df_adj, df_hy_predict], axis=0, sort=True).reset_index(drop=True)
    df_adj_predict['Replacement Count Adjusted US'].fillna(df_adj_predict['Replacement Count Raw'], inplace=True)    

    df_reshape_low = df_adj_predict.pivot(index='Replacement ' + replacement_col, columns='Implant Time', values='Replacement Count Adjusted US').fillna(0)#.astype(int)
    #df_reshape_low = df_reshape_low[df_reshape_low.sum(axis=1) > 0.0001*total]
    df_reshape_low = df_reshape_low[df_reshape_low.index.isin([(str(Y) + 'H' + '{}'.format(h)) for Y in display_year_range for h in range(1, 3)])]
    exec('df_reshape2_hy_'+estimate+'=df_reshape_low.sum(axis=1)')
    
    colors = zip([i for i in np.linspace(0, 1, len(dfos_hy))] + [i for i in np.linspace(0, 1, len(dfos_predict_hy_low))],
                 [0.3]*len(dfos_hy) + [0.8]*len(dfos_predict_hy_low), [0.8]*len(dfos_hy) + [0]*len(dfos_predict_hy_low))

    fig,ax = plt.subplots()
    df_reshape_low.plot(kind='bar', stacked=True, ax=ax, figsize=(20,12), fontsize=20, color=colors)
    ax.grid(axis='y')
    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='both', which='minor', labelsize=16)  
#     if not ylim_replacement:
#         _, ylim_replacement = ax.get_ylim()
#         ylim_replacement = ylim_replacement * 1.3
    #ax.set(ylim=[0, ylim_replacement])
    start, end = ax.get_ylim()
    stepsize = Stepsize_Cal(end) 
    ax.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax.set_xlabel('Expected Replacement Date (in Half Year)', fontsize=20)
    ax.set_ylabel('IPG Counts', fontsize=20)
    ax.legend(list('Sales in ' + df_reshape_low.columns[:len(dfos_hy)]) + list('Predicted New Sales in ' + df_reshape_low.columns[-len(dfos_predict_hy_low):]),
              loc='upper right', bbox_to_anchor=(1.45, 1), fontsize=20)
    
#     ax2 = ax.twinx()
#     (df_reshape_low/total).plot(kind='bar', stacked=True, ax=ax2, figsize=(20,12), fontsize=18, color=colors)
#     #df_reshape_low.plot.bar(stacked=True, ax=ax2, figsize=(20,12), fontsize=18)
#     plt.gca().yaxis.set_major_formatter(PercentFormatter(1))

#     ax2.tick_params(axis='y', labelcolor='red', labelsize=20)
#     #ax2.set(ylim=[0, ylim_replacement/total])
#     start, end = ax2.get_ylim()
#     ax2.yaxis.set_ticks(np.arange(start, end, 0.01))
#     ax2.set_ylabel('Percentage of Total', fontsize=20, color='red')

#     ax.get_legend().remove()
#     ax2.legend(list('Sales in ' + df_reshape_low.columns[:len(dfos_hy)]) + list('Predicted New Sales in ' + df_reshape_low.columns[-len(dfos_predict_hy_low):]),
#                loc='upper right', bbox_to_anchor=(1.45, 1), fontsize=20)

    ax.set_title(r'$\bf{' + f"Replacement\ Timeline\ of\ {total}\ IPGs " + ProgramType + '}$' +
                 ' Based on Low Estimation of Predicted New Sales' + '\n-- ' + 
                 r'$\bf{' + estimate + '}$' +
                 ' expectation based on ' + impedance + ' = ' + imp_est, fontsize=20)
    plt.tick_params(labelright=True)
    #fig.savefig('figures/' + ProgramType + '_replacement_' + estimate + '_estimate_predict_low.png', bbox_inches='tight')
    
    ######### replacement count
    ax_rev.plot(df_reshape_low.index[df_reshape_low.index.isin(display_hy_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_hy_range)], **ls)
    ax_rev.tick_params(axis='both', which='major', labelsize=20)
    ax_rev.tick_params(axis='both', which='minor', labelsize=16)
    #ax_rev.set(ylim=[0, ylim_replacement])
    start, end = ax_rev.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_rev.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_rev.set_xlabel('Time of Sales (in Half Year)', fontsize=20)
    ax_rev.set_ylabel('IPG Counts', fontsize=20)
    
    ax_rev2.plot(df_reshape_low.index[df_reshape_low.index.isin(display_hy_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_hy_range)], **ls)
    ax_rev2.tick_params(axis='y', labelsize=20)
    #ax_rev2.set(ylim=[0, ylim_replacement]); 
    start, end = ax_rev2.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_rev2.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_rev2.set_ylabel('IPG Counts', fontsize=20)

    ######### total count
    if estimate == 'Earliest':
        dfos_hy_low = pd.concat([dfos_hy, dfos_predict_hy_low], axis=0, sort=True).sort_values(['Time in Half Year']).reset_index(drop=True)[['Time in Half Year', 'US Total']]
        dfos_hy_low = dfos_hy_low[dfos_hy_low['Time in Half Year'].isin(df_reshape_low.index[df_reshape_low.index.isin(display_hy_range)].tolist())]
        #ax_trev.plot(dfos_hy_low['Time in Half Year'], dfos_hy_low['US Total'], **dict(ls='dashed', linewidth=3, color='blue', marker='.', markersize=20))

    dfos_total_hy_low = pd.Series(dfos_hy_low['US Total'].tolist(), index=dfos_hy_low['Time in Half Year']).add(
        df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_hy_range)], fill_value=0)
    ax_trev.plot(dfos_total_hy_low.index, dfos_total_hy_low, **tls)
    ax_trev.plot(df_reshape_low.index[df_reshape_low.index.isin(display_hy_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_hy_range)], **ls)
    ax_trev.tick_params(axis='both', which='major', labelsize=20)
    ax_trev.tick_params(axis='both', which='minor', labelsize=16)
    #ax_trev.set(ylim=[0, ylim_replacement])
    start, end = ax_trev.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_trev.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_trev.set_xlabel('Time of Sales (in Half Year)', fontsize=20)
    ax_trev.set_ylabel('IPG Counts', fontsize=20)
    
    ax_trev2.plot(dfos_total_hy_low.index, dfos_total_hy_low, **tls)
    ax_trev2.plot(df_reshape_low.index[df_reshape_low.index.isin(display_hy_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_hy_range)], **ls)
    ax_trev2.tick_params(axis='y', labelsize=20)
    #ax_trev2.set(ylim=[0, ylim_replacement]); 
    start, end = ax_trev2.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_trev2.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_trev2.set_ylabel('IPG Counts', fontsize=20)

ax_rev.grid(axis='y')
lgd = [(fitting + '\n' + estimate + ' expectation of replacement') 
       for fitting in ['Based on low estimation of predicted new sales']
       for estimate in ['& earliest', '& medium', '& latest']]
ax_rev.legend(lgd, loc='center', bbox_to_anchor=(1.4, 0.5), fontsize=16, ncol=1, handleheight=0)    
ax_rev.set_title(('US Orion IPG Replacement Count Prediction ' + ProgramType), fontsize=20, fontweight='bold')
# locs, labels = plt.xticks()
# ax_rev.set_xticklabels(labels, rotation=90)
for tick in ax_rev.get_xticklabels():
    tick.set_rotation(90)
#fig_rev.savefig('figures/' + ProgramType + '_replacement_count_predict_hy.png', bbox_inches='tight')

ax_trev.grid(axis='y')
lgd = [(rev + '\n' + fitting + '\n' + estimate + ' expectation of replacement)')
       for fitting in ['(based on low estimation of predicted new sales']
       for estimate in ['& earliest', '& medium', '& latest']
       for rev in ['New Sales + Replacement', 'Replacement Only']]
ax_trev.legend(lgd, loc='center', bbox_to_anchor=(1.4, 0.42), fontsize=16, ncol=1, handleheight=0)    
ax_trev.set_title(('US Orion Total Count Prediction (New Sales + Replacement) ' + ProgramType), fontsize=20, fontweight='bold')
# locs, labels = plt.xticks()
# ax_trev.set_xticklabels(labels, rotation=90)
for tick in ax_trev.get_xticklabels():
    tick.set_rotation(90)
#fig_trev.savefig('figures/' + ProgramType + '_total_count_predict_hy.png', bbox_inches='tight')

In [ ]:
df_replacement_hy = pd.DataFrame({'Earliest': ((df_reshape_hy_Earliest*battery_replace_ratio).add(
                                                df_reshape2_hy_Earliest*(1-battery_replace_ratio), fill_value=0)).round(),
                                  'Medium': ((df_reshape_hy_Medium*battery_replace_ratio).add(
                                              df_reshape2_hy_Medium*(1-battery_replace_ratio), fill_value=0)).round(),
                                  'Latest': ((df_reshape_hy_Latest*battery_replace_ratio).add(
                                              df_reshape2_hy_Latest*(1-battery_replace_ratio), fill_value=0)).round()})
df_replacement_hy.fillna(0, inplace=True)
df_replacement_hy.sort_index(inplace=True)

In [ ]:
#df_replacement_hy = df_replacement_hy.apply(lambda x: moving_average(x, 3), axis=0)

In [ ]:
replacement_predict_hy = df_replacement_hy.transpose().describe().transpose().sort_index()['mean']
replacement_predict_std_hy = df_replacement_hy.transpose().describe().transpose().sort_index()['std']

In [ ]:
replacement_predict_hy = pd.Series(0, index=display_hy_range).add(replacement_predict_hy, fill_value=0)
replacement_predict_std_hy = pd.Series(0, index=display_hy_range).add(replacement_predict_std_hy, fill_value=0)

In [ ]:
fig,ax = plt.subplots()
fig.set_size_inches(20,10)
ax2 = ax.twinx()
ax.grid(axis='y')
# ax.set_axisbelow(True)
# ax.yaxis.grid(color='gray', linestyle='dashed')

display_skip = 0

ax.plot(replacement_predict_hy.index[replacement_predict_hy.index.isin(display_hy_range)], 
        replacement_predict_hy.values[replacement_predict_hy.index.isin(display_hy_range)],
        linewidth=3, marker='.', markersize=20, color='black')
ax.plot(df_replacement_hy['Earliest'].index[df_replacement_hy['Earliest'].index.isin(display_hy_range)],
        df_replacement_hy['Earliest'].values[df_replacement_hy['Earliest'].index.isin(display_hy_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='blue')
ax.plot(df_replacement_hy['Medium'].index[df_replacement_hy['Medium'].index.isin(display_hy_range)],
        df_replacement_hy['Medium'].values[df_replacement_hy['Medium'].index.isin(display_hy_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='green')
ax.plot(df_replacement_hy['Latest'].index[df_replacement_hy['Latest'].index.isin(display_hy_range)],
        df_replacement_hy['Latest'].values[df_replacement_hy['Latest'].index.isin(display_hy_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='purple')
ax.plot(dfos_r_hy['Time in Half Year'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)][display_skip:],
        dfos_r_hy['US Total'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)][display_skip:],
        linestyle='dotted', marker='.', markersize=10, color='red', alpha=0.6)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time in Half Year', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

ax2.set(ylim=[0, end]); 
ax2.tick_params(axis='y', labelsize=18)
ax2.yaxis.set_ticks(np.arange(0, end, stepsize))
ax2.set_ylabel('Orion IPG Counts', fontsize=20)

for tick in ax.get_xticklabels():
    tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life (average of a, b & c)',
                 '  a) Predicted Replacement (earliest, based on highest impedance)',
                 '  b) Predicted Replacement (medium, based on medium impedance)',
                 '  c) Predicted Replacement (latest, based on lowest impedance)',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)'],
                loc='best', fontsize=20)
#plt.savefig('figures/' + ProgramType + '_IPG_replacement_hy.png')

In [ ]:
fig,ax = plt.subplots()
fig.set_size_inches(20,12)
ax2 = ax.twinx()
ax.grid(axis='y')
# ax.set_axisbelow(True)
# ax.yaxis.grid(color='gray', linestyle='dashed')

display_skip = 0 

ax.plot(replacement_predict_hy.index[replacement_predict_hy.index.isin(display_hy_range)], 
        replacement_predict_hy.values[replacement_predict_hy.index.isin(display_hy_range)],
        linewidth=3, marker='.', markersize=20, color='black')
ax.plot(dfos_r_hy['Time in Half Year'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)][display_skip:],
        dfos_r_hy['US Total'][dfos_r_hy['Time in Half Year'].isin(display_hy_range)][display_skip:],
        linestyle='dotted', marker='.', markersize=10, color='red', alpha=0.6)
ax.errorbar(replacement_predict_hy.index[replacement_predict_hy.index.isin(display_hy_range)], 
            replacement_predict_hy.values[replacement_predict_hy.index.isin(display_hy_range)],
            yerr = replacement_predict_std_hy.values[replacement_predict_std_hy.index.isin(display_hy_range)],
            linewidth=2, color='gray', barsabove=True, capsize=5)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time in Half Year', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

ax2.set(ylim=[0, end]); 
ax2.tick_params(axis='y', labelsize=18)
ax2.yaxis.set_ticks(np.arange(0, end, stepsize))
ax2.set_ylabel('Orion IPG Counts', fontsize=20)

for tick in ax.get_xticklabels():
    tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)',
                 'Range of Predicted Replacement'],
                loc='best', fontsize=20)
#plt.savefig('figures/' + ProgramType + '_IPG_replacement_hy_std.png')

## Results in Quarter

In [ ]:
df['Replacement Date Earliest'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_low']-avg_days_adjust), axis=1)
df['Replacement Date Average'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_medium']-avg_days_adjust), axis=1)
df['Replacement Date Latest'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_high']-avg_days_adjust), axis=1)

df['Replacement Earliest Quarter'] = df['Replacement Date Earliest'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
df['Replacement Average Quarter'] = df['Replacement Date Average'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
df['Replacement Latest Quarter'] = df['Replacement Date Latest'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))

In [ ]:
display_qt_range = [(str(Y) + 'Q' + '{}'.format(q)) for Y in display_year_range0 for q in range(1, 5)]

In [ ]:
for replacement_col, estimate, impedance, imp_est in zip(['Earliest', 'Average', 'Latest'], ['Earliest', 'Medium', 'Latest'], 
                                                         ['highest impedance', 'default impedance', 'lowest impedance'],
                                                         [impedance_high, impedance_default, impedance_low]):

    df_qt = df.groupby(['Implant Time Quarter', 'Replacement ' + replacement_col + ' Quarter']).agg({'Serial Number (hashed)':'count'}).reset_index()
    df_qt.columns = ['Implant Time Quarter', 'Replacement ' + replacement_col + ' Quarter', 'Replacement Count Raw']
    total = df_qt['Replacement Count Raw'].sum()

    df_reshape = df_qt.pivot(index='Replacement ' + replacement_col + ' Quarter', columns='Implant Time Quarter', values='Replacement Count Raw').fillna(0)#.astype(int)
    #df_reshape = df_reshape[df_reshape.sum(axis=1) > 0.0001*total]
    df_reshape = df_reshape[df_reshape.index.isin([(str(Y) + 'Q' + '{}'.format(q)) for Y in display_year_range for q in range(1, 5)])]
    #exec('df_reshape_qt_'+estimate+'=df_reshape.sum(axis=1)')
    exec('df_reshape_qt_'+estimate+'_noforecast=df_reshape.sum(axis=1)')
    
    #colors = zip([i for i in np.linspace(0, 1, len(dfos_qt))], [0.3]*len(dfos_qt), [0.8]*len(dfos_qt))
    np.random.seed(1234567)
    r, g, b = (np.random.rand(3, len(dfos_qt)).tolist())
    colors = zip(r, g, b)
    
    fig,ax = plt.subplots()
    df_reshape.plot(kind='bar', stacked=True, ax=ax, figsize=(20,12), fontsize=18, color=colors)
    ax.grid(axis='y')
    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='both', which='minor', labelsize=16)
    #ax.set(ylim=[0, ylim_replacement/2])
    start, end = ax.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax.set_xlabel('Expected Replacement Date (in Quarter)', fontsize=20)
    ax.set_ylabel('IPG Counts', fontsize=20)
    ax.set_title(r'$\bf{' + f"Replacement\ Timeline\ of\ {total}\ IPGs " + ProgramType + '}$' +
                 '\n-- ' + 
                 r'$\bf{' + estimate + '}$' +
                 ' expectation based on ' + impedance + ' = ' + imp_est, fontsize=20)
    ax.legend('Sales in ' + df_reshape.columns, loc='upper right', bbox_to_anchor=(1.4, 1), fontsize=20);
       
#     ax2 = ax.twinx()
#     (df_reshape/total).plot(kind='bar', stacked=True, ax=ax2, figsize=(20,12), fontsize=18, color=colors)
#     #df_reshape.plot.bar(stacked=True, ax=ax2, figsize=(20,12), fontsize=18)
#     plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
    
#     ax2.tick_params(axis='y', labelcolor='red', labelsize=20)
#     #ax2.set(ylim=[0, ylim_replacement/2/total])
#     start, end = ax2.get_ylim()
#     ax2.yaxis.set_ticks(np.arange(start, end, 0.01))
#     ax2.set_ylabel('Percentage of Total', fontsize=20, color='red')

#     ax.get_legend().remove()
#     ax2.legend('Sales in ' + df_reshape.columns, loc='upper right', bbox_to_anchor=(1.4, 1), fontsize=20);
    
    plt.tick_params(labelright=True)
    #plt.savefig('figures/' + ProgramType + '_replacement_' + estimate + '_estimate_qt.png', bbox_inches='tight')

In [ ]:
for replacement_col, estimate, impedance, imp_est in zip(['Earliest', 'Average', 'Latest'], ['Earliest', 'Medium', 'Latest'], 
                                                         ['highest impedance', 'default impedance', 'lowest impedance'],
                                                         [impedance_high, impedance_default, impedance_low]):

    df_qt = df.groupby(['Implant Time Quarter', 'Replacement ' + replacement_col + ' Quarter']).agg({'Serial Number (hashed)':'count'}).reset_index()
    df_qt.columns = ['Implant Time Quarter', 'Replacement ' + replacement_col + ' Quarter', 'Replacement Count Raw']
    total = df_implant_quarter['US Total'].sum()

    df_adj = pd.merge(df_qt, df_implant_quarter, on='Implant Time Quarter', how='left')
    df_adj['Replacement Count Adjusted US'] = (df_adj['Replacement Count Raw'] * df_adj['US Total']/df_adj['Partial Count'])
    df_adj['Replacement Count Adjusted US'] = df_adj['Replacement Count Adjusted US'].replace([np.inf, -np.inf, np.nan], 0).round().astype(int)

    df_reshape = df_adj.pivot(index='Replacement ' + replacement_col + ' Quarter', columns='Implant Time Quarter', values='Replacement Count Adjusted US').fillna(0)#.astype(int)
    #df_reshape = df_reshape[df_reshape.sum(axis=1) > 0.0001*total]
    df_reshape = df_reshape[df_reshape.index.isin([(str(Y) + 'Q' + '{}'.format(q)) for Y in display_year_range for q in range(1, 5)])]
    #exec('df_reshape_qt_'+estimate+'=df_reshape.sum(axis=1)')
    exec('df_reshape_qt_'+estimate+'_noforecast=df_reshape.sum(axis=1)')
    
    #colors = zip([i for i in np.linspace(0, 1, len(dfos_qt))], [0.3]*len(dfos_qt), [0.8]*len(dfos_qt))
    np.random.seed(1234567)
    r, g, b = (np.random.rand(3, len(dfos_qt)).tolist())
    colors = zip(r, g, b)
    
    fig,ax = plt.subplots()
    df_reshape.plot(kind='bar', stacked=True, ax=ax, figsize=(20,12), fontsize=18, color=colors)
    ax.grid(axis='y')
    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='both', which='minor', labelsize=16)
    #ax.set(ylim=[0, ylim_replacement/2])
    start, end = ax.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax.set_xlabel('Expected Replacement Date (in Quarter)', fontsize=20)
    ax.set_ylabel('IPG Counts', fontsize=20)
    ax.set_title(r'$\bf{' + f"Replacement\ Timeline\ of\ {total}\ IPGs " + ProgramType + '}$' +
                 ', Adjusted by US Total Sales' + '\n-- ' + 
                 r'$\bf{' + estimate + '}$' +
                 ' expectation based on ' + impedance + ' = ' + imp_est, fontsize=20)
    ax.legend('Sales in ' + df_reshape.columns, loc='upper right', bbox_to_anchor=(1.4, 1), fontsize=20);

#     ax2 = ax.twinx()
#     (df_reshape/total).plot(kind='bar', stacked=True, ax=ax2, figsize=(20,12), fontsize=18, color=colors)
#     #df_reshape.plot.bar(stacked=True, ax=ax2, figsize=(20,12), fontsize=18)
#     plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
    
#     ax2.tick_params(axis='y', labelcolor='red', labelsize=20)
#     #ax2.set(ylim=[0, ylim_replacement/2/total])
#     start, end = ax2.get_ylim()
#     ax2.yaxis.set_ticks(np.arange(start, end, 0.01))
#     ax2.set_ylabel('Percentage of Total', fontsize=20, color='red')

#     ax.get_legend().remove()
#     ax2.legend('Sales in ' + df_reshape.columns, loc='upper right', bbox_to_anchor=(1.4, 1), fontsize=20);
    
    plt.tick_params(labelright=True)
    #plt.savefig('figures/' + ProgramType + '_replacement_' + estimate + '_estimate_qt.png', bbox_inches='tight')

In [ ]:
random_seed=1234567
df_predict_low = []
df_ = df[['Serial Number (hashed)', 'Longevity_low', 'Longevity_medium', 'Longevity_high']]
for pred_date, pred_count, pred_date1 in zip(list(dfos_qt['Time in Quarter'])[-1:] + list(dfos_predict_qt_low['Time in Quarter']),
                                             list(dfos_qt['US Total'])[-1:] + list(dfos_predict_qt_low['US Total']),
    [(str(Y) + '{}'.format(q)) for Y in predict_year_range for q in ['-02-16', '-05-17', '-08-16', '-11-16']][1:]):

    df_tmp = df_.sample(n=pred_count, replace=True, random_state=random_seed+pred_count)
    df_tmp['Serial Number (hashed)'] = 'Predicted'

    df_tmp['Implant Date'] = pd.to_datetime(pred_date1, format='%Y-%m-%d')
    df_tmp['Replacement Date Earliest'] = df_tmp.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_low']-avg_days_adjust), axis=1)
    df_tmp['Replacement Date Average'] = df_tmp.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_medium']-avg_days_adjust), axis=1)
    df_tmp['Replacement Date Latest'] = df_tmp.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_high']-avg_days_adjust), axis=1)
    df_tmp['Implant Time Quarter'] = pred_date
    df_tmp['Replacement Earliest Quarter'] = df_tmp['Replacement Date Earliest'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
    df_tmp['Replacement Average Quarter'] = df_tmp['Replacement Date Average'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
    df_tmp['Replacement Latest Quarter'] = df_tmp['Replacement Date Latest'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
    df_predict_low.append(df_tmp)
    
df_predict_low = pd.concat(df_predict_low, axis=0, sort=True).reset_index(drop=True)

# df_predict_high = []
# for pred_date, pred_count, pred_date1 in zip(dfos_predict_qt_high['Time in Quarter'], dfos_predict_qt_high['US Total'],
#     [(str(Y) + '{}'.format(q)) for Y in predict_year_range for q in ['-02-16', '-05-17', '-08-16', '-11-16']]):

#     df_tmp = df_.sample(n=pred_count, replace=True, random_state=random_seed+pred_count)
#     df_tmp['Serial Number (hashed)'] = 'Predicted'

#     df_tmp['Implant Date'] = pd.to_datetime(pred_date1, format='%Y-%m-%d')
#     df_tmp['Replacement Date Earliest'] = df_tmp.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_low']-avg_days_adjust), axis=1)
#     df_tmp['Replacement Date Average'] = df_tmp.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_medium']-avg_days_adjust), axis=1)
#     df_tmp['Replacement Date Latest'] = df_tmp.apply(lambda x: x['Implant Date'] + timedelta(days=365*x['Longevity_high']-avg_days_adjust), axis=1)
#     df_tmp['Implant Time Quarter'] = pred_date
#     df_tmp['Replacement Earliest Quarter'] = df_tmp['Replacement Date Earliest'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
#     df_tmp['Replacement Average Quarter'] = df_tmp['Replacement Date Average'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
#     df_tmp['Replacement Latest Quarter'] = df_tmp['Replacement Date Latest'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
#     df_predict_high.append(df_tmp)
    
# df_predict_high = pd.concat(df_predict_high, axis=0, sort=True).reset_index(drop=True)

In [ ]:
fig_rev,ax_rev = plt.subplots()
fig_rev.set_size_inches(16,11)
ax_rev2 = ax_rev.twinx()

fig_trev,ax_trev = plt.subplots()
fig_trev.set_size_inches(16,11)
ax_trev2 = ax_trev.twinx()
############ low estimate for predicted sales
for replacement_col, estimate, impedance, imp_est, ls, tls in zip(['Earliest', 'Average', 'Latest'],
                                                 ['Earliest', 'Medium', 'Latest'], 
                                                 ['highest impedance', 'default impedance', 'lowest impedance'],
                                                 [impedance_high, impedance_default, impedance_low],
                                                 [dict(linewidth=2, ls='dotted', color='gray', marker='.', markersize=10),
                                                  dict(linewidth=3, color='blue', marker='.', markersize=20),
                                                  dict(linewidth=2, ls='dashed', color='gray', marker='.', markersize=10)],
                                                 [dict(linewidth=4, ls='dotted', color='gray', marker='*', markersize=10),
                                                  dict(linewidth=5, color='blue', marker='*', markersize=20),
                                                  dict(linewidth=4, ls='dashed', color='gray', marker='*', markersize=10)]):

    df_qt = df.groupby(['Implant Time Quarter', 'Replacement ' + replacement_col + ' Quarter']).agg({'Serial Number (hashed)':'count'}).reset_index()
    df_qt.columns = ['Implant Time Quarter', 'Replacement ' + replacement_col + ' Quarter', 'Replacement Count Raw']

    df_qt_predict = df_predict_low.groupby(['Implant Time Quarter', 'Replacement ' + replacement_col + ' Quarter']).agg({'Serial Number (hashed)':'count'}).reset_index()
    df_qt_predict.columns = ['Implant Time Quarter', 'Replacement ' + replacement_col + ' Quarter', 'Replacement Count Raw']
    total = df_implant_quarter['US Total'].sum() + df_qt_predict['Replacement Count Raw'].sum()

    df_adj = pd.merge(df_qt, df_implant_quarter, on='Implant Time Quarter', how='left')
    df_adj['Replacement Count Adjusted US'] = (df_adj['Replacement Count Raw'] * df_adj['US Total']/df_adj['Partial Count'])
    df_adj['Replacement Count Adjusted US'] = df_adj['Replacement Count Adjusted US'].replace([np.inf, -np.inf, np.nan], 0).round().astype(int)

    df_adj_predict = pd.concat([df_adj, df_qt_predict], axis=0, sort=True).reset_index(drop=True)
    df_adj_predict['Replacement Count Adjusted US'].fillna(df_adj_predict['Replacement Count Raw'], inplace=True)    

    df_reshape_low = df_adj_predict.pivot(index='Replacement ' + replacement_col + ' Quarter', columns='Implant Time Quarter', values='Replacement Count Adjusted US').fillna(0)#.astype(int)
    #df_reshape_low = df_reshape_low[df_reshape_low.sum(axis=1) > 0.0001*total]
    df_reshape_low = df_reshape_low[df_reshape_low.index.isin([(str(Y) + 'Q' + '{}'.format(q)) for Y in display_year_range for q in range(1, 5)])]
    exec('df_reshape_qt_'+estimate+'=df_reshape_low.sum(axis=1)')
    
    colors = zip([i for i in np.linspace(0, 1, len(dfos_qt))] + [i for i in np.linspace(0, 1, len(dfos_predict_qt_low))],
            [0.3]*(len(dfos_qt)) + [0.8]*len(dfos_predict_qt_low), [0.8]*(len(dfos_qt)) + [0]*len(dfos_predict_qt_low))

    fig,ax = plt.subplots()
    df_reshape_low.plot(kind='bar', stacked=True, ax=ax, figsize=(20,12), fontsize=20, color=colors)
    ax.grid(axis='y')
    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='both', which='minor', labelsize=16)  
#     if not ylim_replacement:
#         _, ylim_replacement = ax.get_ylim()
#         ylim_replacement = ylim_replacement * 1.3
    #ax.set(ylim=[0, ylim_replacement])
    start, end = ax.get_ylim()
    stepsize = Stepsize_Cal(end) 
    ax.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax.set_xlabel('Expected Replacement Date (in Quarter)', fontsize=20)
    ax.set_ylabel('IPG Counts', fontsize=20)
    ax.legend(list('Sales in ' + df_reshape_low.columns[:len(dfos_qt)]) + list('Predicted Sales in ' + df_reshape_low.columns[-len(dfos_predict_qt_low)::]),
              loc='upper right', bbox_to_anchor=(1.45, 1), fontsize=20)
    
#     ax2 = ax.twinx()
#     (df_reshape_low/total).plot(kind='bar', stacked=True, ax=ax2, figsize=(20,12), fontsize=18, color=colors)
#     #df_reshape_low.plot.bar(stacked=True, ax=ax2, figsize=(20,12), fontsize=18)
#     plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
    
#     ax2.tick_params(axis='y', labelcolor='red', labelsize=20)
#     #ax2.set(ylim=[0, ylim_replacement/total])
#     start, end = ax2.get_ylim()
#     ax2.yaxis.set_ticks(np.arange(start, end, 0.01))
#     ax2.set_ylabel('Percentage of Total', fontsize=20, color='red')
    
#     ax.get_legend().remove()
#     ax2.legend(list('Sales in ' + df_reshape_low.columns[:len(dfos_qt)]) + list('Predicted Sales in ' + df_reshape_low.columns[-len(dfos_predict_qt_low)::]),
#                loc='upper right', bbox_to_anchor=(1.45, 1), fontsize=20)

    ax.set_title(r'$\bf{' + f"Replacement\ Timeline\ of\ {total}\ IPGs " + ProgramType + '}$' +
                 ' Based on Low Estimation of Predicted New Sales' + '\n-- ' + 
                 r'$\bf{' + estimate + '}$' +
                 ' expectation based on ' + impedance + ' = ' + imp_est, fontsize=20)
    plt.tick_params(labelright=True)
    #fig.savefig('figures/' + ProgramType + '_replacement_' + estimate + '_estimate_predict_low.png', bbox_inches='tight')
    
    ######### replacement count
    ax_rev.plot(df_reshape_low.index[df_reshape_low.index.isin(display_qt_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_qt_range)], **ls)
    ax_rev.tick_params(axis='both', which='major', labelsize=20)
    ax_rev.tick_params(axis='both', which='minor', labelsize=16)
    #ax_rev.set(ylim=[0, ylim_replacement])
    start, end = ax_rev.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_rev.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_rev.set_xlabel('Time of Sales (in Quarter)', fontsize=20)
    ax_rev.set_ylabel('IPG Counts', fontsize=20)
    
    ax_rev2.plot(df_reshape_low.index[df_reshape_low.index.isin(display_qt_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_qt_range)], **ls)
    ax_rev2.tick_params(axis='y', labelsize=20)
    #ax_rev2.set(ylim=[0, ylim_replacement]); 
    start, end = ax_rev2.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_rev2.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_rev2.set_ylabel('IPG Counts', fontsize=20)

    ######### total count
    if estimate == 'Earliest':
        dfos_qt_low = pd.concat([dfos_qt, dfos_predict_qt_low], axis=0, sort=True).sort_values(['Time in Quarter']).reset_index(drop=True)[['Time in Quarter', 'US Total']]
        dfos_qt_low = dfos_qt_low[dfos_qt_low['Time in Quarter'].isin(df_reshape_low.index[df_reshape_low.index.isin(display_qt_range)].tolist())]
        #ax_trev.plot(dfos_qt_low['Time in Quarter'], dfos_qt_low['US Total'], **dict(ls='dashed', linewidth=3, color='blue', marker='.', markersize=20))

    dfos_total_qt_low = pd.Series(dfos_qt_low['US Total'].tolist(), index=dfos_qt_low['Time in Quarter']).add(
        df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_qt_range)], fill_value=0)
    ax_trev.plot(dfos_total_qt_low.index, dfos_total_qt_low, **tls)
    ax_trev.plot(df_reshape_low.index[df_reshape_low.index.isin(display_qt_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_qt_range)], **ls)
    ax_trev.tick_params(axis='both', which='major', labelsize=20)
    ax_trev.tick_params(axis='both', which='minor', labelsize=16)
    #ax_trev.set(ylim=[0, ylim_replacement])
    start, end = ax_trev.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_trev.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_trev.set_xlabel('Time of Sales (in Quarter)', fontsize=20)
    ax_trev.set_ylabel('IPG Counts', fontsize=20)
    
    ax_trev2.plot(dfos_total_qt_low.index, dfos_total_qt_low, **tls)
    ax_trev2.plot(df_reshape_low.index[df_reshape_low.index.isin(display_qt_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_qt_range)], **ls)
    ax_trev2.tick_params(axis='y', labelsize=20)
    #ax_trev2.set(ylim=[0, ylim_replacement]); 
    start, end = ax_trev2.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_trev2.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_trev2.set_ylabel('IPG Counts', fontsize=20)


ax_rev.grid(axis='y')
lgd = [(fitting + '\n' + estimate + ' expectation of replacement') 
       for fitting in ['Based on low estimation of predicted new sales']
       for estimate in ['& earliest', '& medium', '& latest']]
ax_rev.legend(lgd, loc='center', bbox_to_anchor=(1.4, 0.5), fontsize=16, ncol=1, handleheight=0)    
ax_rev.set_title(('US Orion IPG Replacement Count Prediction ' + ProgramType), fontsize=20, fontweight='bold')
# locs, labels = plt.xticks()
# ax_rev.set_xticklabels(labels, rotation=90)
for tick in ax_rev.get_xticklabels():
    tick.set_rotation(90)
#fig_rev.savefig('figures/' + ProgramType + '_replacement_count_predict_qt.png', bbox_inches='tight')

ax_trev.grid(axis='y')
lgd = [(rev + '\n' + fitting + '\n' + estimate + ' expectation of replacement)')
       for fitting in ['(based on low estimation of predicted new sales']
       for estimate in ['& earliest', '& medium', '& latest']
       for rev in ['New Sales + Replacement', 'Replacement Only']]
ax_trev.legend(lgd, loc='center', bbox_to_anchor=(1.4, 0.42), fontsize=16, ncol=1, handleheight=0)    
ax_trev.set_title(('US Orion Total Count Prediction (New Sales + Replacement) ' + ProgramType), fontsize=20, fontweight='bold')
# locs, labels = plt.xticks()
# ax_trev.set_xticklabels(labels, rotation=90)
for tick in ax_trev.get_xticklabels():
    tick.set_rotation(90)
#fig_trev.savefig('figures/' + ProgramType + '_total_count_predict_qt.png', bbox_inches='tight')

In [ ]:
df_replacement_qt = pd.DataFrame({'Earliest': df_reshape_qt_Earliest,
                                  'Medium': df_reshape_qt_Medium,
                                  'Latest': df_reshape_qt_Latest})
df_replacement_qt.fillna(0, inplace=True)
df_replacement_qt.sort_index(inplace=True)

In [ ]:
df_replacement_qt_noforecast = pd.DataFrame({'Earliest': df_reshape_qt_Earliest_noforecast,
                                             'Medium': df_reshape_qt_Medium_noforecast,
                                             'Latest': df_reshape_qt_Latest_noforecast})
df_replacement_qt_noforecast.fillna(0, inplace=True)
df_replacement_qt_noforecast.sort_index(inplace=True)

In [ ]:
replacement_predict_qt = df_replacement_qt.transpose().describe().transpose().sort_index()['mean']
replacement_predict_std_qt = df_replacement_qt.transpose().describe().transpose().sort_index()['std']

replacement_predict_qt_noforecast = df_replacement_qt_noforecast.transpose().describe().transpose().sort_index()['mean']
replacement_predict_std_qt_noforecast = df_replacement_qt_noforecast.transpose().describe().transpose().sort_index()['std']

In [ ]:
replacement_predict_qt = pd.Series(0, index=display_qt_range).add(replacement_predict_qt, fill_value=0)
replacement_predict_std_qt = pd.Series(0, index=display_qt_range).add(replacement_predict_std_qt, fill_value=0)

replacement_predict_qt_noforecast = pd.Series(0, index=display_qt_range).add(replacement_predict_qt_noforecast, fill_value=0)
replacement_predict_std_qt_noforecast = pd.Series(0, index=display_qt_range).add(replacement_predict_std_qt_noforecast, fill_value=0)

In [ ]:
# without sales forecast
fig,ax = plt.subplots()
fig.set_size_inches(20,12)
ax2 = ax.twinx()
#plt.grid(axis='y')
ax.grid(axis='y')
# ax.set_axisbelow(True)
# ax.yaxis.grid(color='gray', linestyle='dashed')

display_skip = 0

ax.plot(replacement_predict_qt_noforecast.index[replacement_predict_qt_noforecast.index.isin(display_qt_range)], 
        replacement_predict_qt_noforecast.values[replacement_predict_qt_noforecast.index.isin(display_qt_range)],
        linewidth=3, marker='.', markersize=20, color='black')
ax.plot(df_replacement_qt_noforecast['Earliest'].index[df_replacement_qt_noforecast['Earliest'].index.isin(display_qt_range)],
        df_replacement_qt_noforecast['Earliest'].values[df_replacement_qt_noforecast['Earliest'].index.isin(display_qt_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='blue')
ax.plot(df_replacement_qt_noforecast['Medium'].index[df_replacement_qt_noforecast['Medium'].index.isin(display_qt_range)],
        df_replacement_qt_noforecast['Medium'].values[df_replacement_qt_noforecast['Medium'].index.isin(display_qt_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='green')
ax.plot(df_replacement_qt_noforecast['Latest'].index[df_replacement_qt_noforecast['Latest'].index.isin(display_qt_range)],
        df_replacement_qt_noforecast['Latest'].values[df_replacement_qt_noforecast['Latest'].index.isin(display_qt_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='purple')
ax.plot(dfos_r_qt['Time in Quarter'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)][display_skip:],
        dfos_r_qt['US Total'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)][display_skip:],
        linestyle='dotted', marker='.', markersize=10, color='red', alpha=0.6)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time (in Quarter)', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

ax2.set(ylim=[0, end]); 
ax2.tick_params(axis='y', labelsize=18)
ax2.yaxis.set_ticks(np.arange(0, end, stepsize))
ax2.set_ylabel('Orion IPG Counts', fontsize=20)

for tick in ax.get_xticklabels():
    tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life (average of a, b & c)',
                 '  a) Predicted Replacement (earliest, based on highest impedance)',
                 '  b) Predicted Replacement (medium, based on medium impedance)',
                 '  c) Predicted Replacement (latest, based on lowest impedance)',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)'],
                loc='best', fontsize=20)
plt.savefig('figures/' + ProgramType + '_IPG_replacement_qt_noforecast.png')

In [ ]:
# without sales forecast
fig,ax = plt.subplots()
fig.set_size_inches(20,12)
ax2 = ax.twinx()
#plt.grid(axis='y')
ax.grid(axis='y')
# ax.set_axisbelow(True)
# ax.yaxis.grid(color='gray', linestyle='dashed')

display_skip = 0 

ax.plot(replacement_predict_qt_noforecast.index[replacement_predict_qt_noforecast.index.isin(display_qt_range)], 
        replacement_predict_qt_noforecast.values[replacement_predict_qt_noforecast.index.isin(display_qt_range)],
        linewidth=3, marker='.', markersize=20, color='black')
ax.plot(dfos_r_qt['Time in Quarter'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)][display_skip:],
        dfos_r_qt['US Total'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)][display_skip:],
        linestyle='dotted', marker='.', markersize=10, color='red', alpha=0.6)
ax.errorbar(replacement_predict_qt_noforecast.index[replacement_predict_qt_noforecast.index.isin(display_qt_range)], 
            replacement_predict_qt_noforecast.values[replacement_predict_qt_noforecast.index.isin(display_qt_range)],
            yerr = replacement_predict_std_qt_noforecast.values[replacement_predict_std_qt_noforecast.index.isin(display_qt_range)],
            linewidth=2, color='gray', barsabove=True, capsize=5)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time (in Quarter)', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

ax2.set(ylim=[0, end]); 
ax2.tick_params(axis='y', labelsize=18)
ax2.yaxis.set_ticks(np.arange(0, end, stepsize))
ax2.set_ylabel('Orion IPG Counts', fontsize=20)

for tick in ax.get_xticklabels():
    tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)',
                 'Range of Predicted Replacement'],
                loc='best', fontsize=20)
#plt.savefig('figures/' + ProgramType + '_IPG_replacement_qt_std_noforecast.png')

In [ ]:
# without sales forecast
fig,ax = plt.subplots()
fig.set_size_inches(20,12)
ax.grid(axis='y')
ax.set_axisbelow(True)
ax.yaxis.grid(color='gray', linestyle='dashed')

width = 0.8

bars1 = replacement_predict_qt_noforecast.values[replacement_predict_qt_noforecast.index.isin(display_qt_range)]
std1 = replacement_predict_std_qt_noforecast.values[replacement_predict_std_qt_noforecast.index.isin(display_qt_range)]
bars2 = np.array(dfos_r_qt['US Total'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)])
X1 = replacement_predict_qt_noforecast.index[replacement_predict_qt_noforecast.index.isin(display_qt_range)]
X2 = dfos_r_qt['Time in Quarter'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)]
_X1 = np.arange(len(X1))
_X2 = np.arange(len(X2))
ax.bar(_X1 - width/2, bars1, width=width/2, align="edge", color='black',
       yerr=std1, capsize=5)
ax.bar(_X2, bars2, width=width/2, align="edge", color='red', alpha=0.4)
# for i in range(min(len(X1),len(X2))):
#     ax.text(x = _X1[i] - width/2 - width/5, y = bars1[i] + max(bars1)*0.03, s = '{:.1f}%'.format(bars1[i]/bars2[i]*100), size=16)
for i in range(len(X1)):
    ax.text(x = _X1[i] - width/2 - width/10, y = bars1[i] + max(bars1)*0.03, s = '{:.0f}'.format(round(bars1[i])), size=16)
for i in range(len(X2)):
    ax.text(x = _X2[i] - width/10, y = bars2[i] + max(bars2)*0.03, s = '{}'.format(bars2[i]), size=16)
plt.xticks(_X1, X1)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time (in Quarter)', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

for tick in ax.get_xticklabels():
    tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType +
              '\nPredicted Replacement (due to Battery End of Life) vs. Actual Total Replacements'), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life (bar: prediction range)',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)'],
                loc='best', fontsize=20)
plt.tick_params(labelright=True)
plt.savefig('figures/' + ProgramType + '_IPG_replacement_predict_qt_noforecast.png')

In [ ]:
fig,ax = plt.subplots()
fig.set_size_inches(20,12)
ax2 = ax.twinx()
#plt.grid(axis='y')
ax.grid(axis='y')
# ax.set_axisbelow(True)
# ax.yaxis.grid(color='gray', linestyle='dashed')

display_skip = 0

ax.plot(replacement_predict_qt.index[replacement_predict_qt.index.isin(display_qt_range)], 
        replacement_predict_qt.values[replacement_predict_qt.index.isin(display_qt_range)],
        linewidth=3, marker='.', markersize=20, color='black')
ax.plot(df_replacement_qt['Earliest'].index[df_replacement_qt['Earliest'].index.isin(display_qt_range)],
        df_replacement_qt['Earliest'].values[df_replacement_qt['Earliest'].index.isin(display_qt_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='blue')
ax.plot(df_replacement_qt['Medium'].index[df_replacement_qt['Medium'].index.isin(display_qt_range)],
        df_replacement_qt['Medium'].values[df_replacement_qt['Medium'].index.isin(display_qt_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='green')
ax.plot(df_replacement_qt['Latest'].index[df_replacement_qt['Latest'].index.isin(display_qt_range)],
        df_replacement_qt['Latest'].values[df_replacement_qt['Latest'].index.isin(display_qt_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='purple')
ax.plot(dfos_r_qt['Time in Quarter'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)][display_skip:],
        dfos_r_qt['US Total'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)][display_skip:],
        linestyle='dotted', marker='.', markersize=10, color='red', alpha=0.6)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time (in Quarter)', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

ax2.set(ylim=[0, end]); 
ax2.tick_params(axis='y', labelsize=18)
ax2.yaxis.set_ticks(np.arange(0, end, stepsize))
ax2.set_ylabel('Orion IPG Counts', fontsize=20)

for tick in ax.get_xticklabels():
    tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life (average of a, b & c)',
                 '  a) Predicted Replacement (earliest, based on highest impedance)',
                 '  b) Predicted Replacement (medium, based on medium impedance)',
                 '  c) Predicted Replacement (latest, based on lowest impedance)',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)'],
                loc='best', fontsize=20)
#plt.savefig('figures/' + ProgramType + '_IPG_replacement_qt.png')

In [ ]:
fig,ax = plt.subplots()
fig.set_size_inches(20,12)
ax2 = ax.twinx()
#plt.grid(axis='y')
ax.grid(axis='y')
# ax.set_axisbelow(True)
# ax.yaxis.grid(color='gray', linestyle='dashed')

display_skip = 0

ax.plot(replacement_predict_qt.index[replacement_predict_qt.index.isin(display_qt_range)], 
        replacement_predict_qt.values[replacement_predict_qt.index.isin(display_qt_range)],
        linewidth=3, marker='.', markersize=20, color='black')
ax.plot(dfos_r_qt['Time in Quarter'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)][display_skip:],
        dfos_r_qt['US Total'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)][display_skip:],
        linestyle='dotted', marker='.', markersize=10, color='red', alpha=0.6)
ax.errorbar(replacement_predict_qt.index[replacement_predict_qt.index.isin(display_qt_range)], 
            replacement_predict_qt.values[replacement_predict_qt.index.isin(display_qt_range)],
            yerr = replacement_predict_std_qt.values[replacement_predict_std_qt.index.isin(display_qt_range)],
            linewidth=2, color='gray', barsabove=True, capsize=5)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time (in Quarter)', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

ax2.set(ylim=[0, end]); 
ax2.tick_params(axis='y', labelsize=18)
ax2.yaxis.set_ticks(np.arange(0, end, stepsize))
ax2.set_ylabel('Orion IPG Counts', fontsize=20)

for tick in ax.get_xticklabels():
    tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)',
                 'Range of Predicted Replacement'],
                loc='best', fontsize=20)
#plt.savefig('figures/' + ProgramType + '_IPG_replacement_qt_std.png')

In [ ]:
fig,ax = plt.subplots()
fig.set_size_inches(20,12)
ax.grid(axis='y')
ax.set_axisbelow(True)
ax.yaxis.grid(color='gray', linestyle='dashed')

width = 0.8

bars1 = replacement_predict_qt.values[replacement_predict_qt.index.isin(display_qt_range)]
std1 = replacement_predict_std_qt.values[replacement_predict_std_qt.index.isin(display_qt_range)]
bars2 = np.array(dfos_r_qt['US Total'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)])
X1 = replacement_predict_qt.index[replacement_predict_qt.index.isin(display_qt_range)]
X2 = dfos_r_qt['Time in Quarter'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)]
_X1 = np.arange(len(X1))
_X2 = np.arange(len(X2))
ax.bar(_X1 - width/2, bars1, width=width/2, align="edge", color='black',
       yerr=std1, capsize=5)
ax.bar(_X2, bars2, width=width/2, align="edge", color='red', alpha=0.4)
# for i in range(min(len(X1),len(X2))):
#     ax.text(x = _X1[i] - width/2 - width/5, y = bars1[i] + max(bars1)*0.03, s = '{:.1f}%'.format(bars1[i]/bars2[i]*100), size=16)
for i in range(len(X1)):
    ax.text(x = _X1[i] - width/2 - width/10, y = bars1[i] + max(bars1)*0.03, s = '{:.0f}'.format(round(bars1[i])), size=16)
for i in range(len(X2)):
    ax.text(x = _X2[i] - width/10, y = bars2[i] + max(bars2)*0.03, s = '{}'.format(bars2[i]), size=16)
plt.xticks(_X1, X1)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.2
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time (in Quarter)', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

for tick in ax.get_xticklabels():
    tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType +
              '\nPredicted Replacement (due to Battery End of Life) vs. Actual Total Replacements'), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life (bar: prediction range)',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)'],
                loc='best', fontsize=20)
plt.tick_params(labelright=True)
#plt.savefig('figures/' + ProgramType + '_IPG_replacement_predict_qt.png')

### Adjust based on all causes for replacement

In [ ]:
df['Replacement Date Earliest'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=avg_days_replace), axis=1)
df['Replacement Date Average'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=avg_days_replace), axis=1)
df['Replacement Date Latest'] = df.apply(lambda x: x['Implant Date'] + timedelta(days=avg_days_replace), axis=1)

df['Replacement Earliest Quarter'] = df['Replacement Date Earliest'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
df['Replacement Average Quarter'] = df['Replacement Date Average'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
df['Replacement Latest Quarter'] = df['Replacement Date Latest'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))

In [ ]:
df_predict_low['Replacement Date Earliest'] = df_predict_low.apply(lambda x: x['Implant Date'] + timedelta(days=avg_days_replace), axis=1)
df_predict_low['Replacement Date Average'] = df_predict_low.apply(lambda x: x['Implant Date'] + timedelta(days=avg_days_replace), axis=1)
df_predict_low['Replacement Date Latest'] = df_predict_low.apply(lambda x: x['Implant Date'] + timedelta(days=avg_days_replace), axis=1)
df_predict_low['Replacement Earliest Quarter'] = df_predict_low['Replacement Date Earliest'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
df_predict_low['Replacement Average Quarter'] = df_predict_low['Replacement Date Average'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))
df_predict_low['Replacement Latest Quarter'] = df_predict_low['Replacement Date Latest'].apply(lambda x: str(x.year) + 'Q' + str(math.ceil(x.month/3)))

In [ ]:
fig_rev,ax_rev = plt.subplots()
fig_rev.set_size_inches(16,11)
ax_rev2 = ax_rev.twinx()

fig_trev,ax_trev = plt.subplots()
fig_trev.set_size_inches(16,11)
ax_trev2 = ax_trev.twinx()
############ low estimate for predicted sales
for replacement_col, estimate, impedance, imp_est, ls, tls in zip(['Earliest', 'Average', 'Latest'],
                                                 ['Earliest', 'Medium', 'Latest'], 
                                                 ['highest impedance', 'default impedance', 'lowest impedance'],
                                                 [impedance_high, impedance_default, impedance_low],
                                                 [dict(linewidth=2, ls='dotted', color='gray', marker='.', markersize=10),
                                                  dict(linewidth=3, color='blue', marker='.', markersize=20),
                                                  dict(linewidth=2, ls='dashed', color='gray', marker='.', markersize=10)],
                                                 [dict(linewidth=4, ls='dotted', color='gray', marker='*', markersize=10),
                                                  dict(linewidth=5, color='blue', marker='*', markersize=20),
                                                  dict(linewidth=4, ls='dashed', color='gray', marker='*', markersize=10)]):

    df_qt = df.groupby(['Implant Time Quarter', 'Replacement ' + replacement_col + ' Quarter']).agg({'Serial Number (hashed)':'count'}).reset_index()
    df_qt.columns = ['Implant Time Quarter', 'Replacement ' + replacement_col + ' Quarter', 'Replacement Count Raw']

    df_qt_predict = df_predict_low.groupby(['Implant Time Quarter', 'Replacement ' + replacement_col + ' Quarter']).agg({'Serial Number (hashed)':'count'}).reset_index()
    df_qt_predict.columns = ['Implant Time Quarter', 'Replacement ' + replacement_col + ' Quarter', 'Replacement Count Raw']
    total = df_implant_quarter['US Total'].sum() + df_qt_predict['Replacement Count Raw'].sum()

    df_adj = pd.merge(df_qt, df_implant_quarter, on='Implant Time Quarter', how='left')
    df_adj['Replacement Count Adjusted US'] = (df_adj['Replacement Count Raw'] * df_adj['US Total']/df_adj['Partial Count'])
    df_adj['Replacement Count Adjusted US'] = df_adj['Replacement Count Adjusted US'].replace([np.inf, -np.inf, np.nan], 0).round().astype(int)

    df_adj_predict = pd.concat([df_adj, df_qt_predict], axis=0, sort=True).reset_index(drop=True)
    df_adj_predict['Replacement Count Adjusted US'].fillna(df_adj_predict['Replacement Count Raw'], inplace=True)    

    df_reshape_low = df_adj_predict.pivot(index='Replacement ' + replacement_col + ' Quarter', columns='Implant Time Quarter', values='Replacement Count Adjusted US').fillna(0)#.astype(int)
    #df_reshape_low = df_reshape_low[df_reshape_low.sum(axis=1) > 0.0001*total]
    df_reshape_low = df_reshape_low[df_reshape_low.index.isin([(str(Y) + 'Q' + '{}'.format(q)) for Y in display_year_range for q in range(1, 5)])]
    exec('df_reshape2_qt_'+estimate+'=df_reshape_low.sum(axis=1)')
    
    colors = zip([i for i in np.linspace(0, 1, len(dfos_qt))] + [i for i in np.linspace(0, 1, len(dfos_predict_qt_low))],
            [0.3]*(len(dfos_qt)) + [0.8]*len(dfos_predict_qt_low), [0.8]*(len(dfos_qt)) + [0]*len(dfos_predict_qt_low))

    fig,ax = plt.subplots()
    df_reshape_low.plot(kind='bar', stacked=True, ax=ax, figsize=(20,12), fontsize=20, color=colors)
    ax.grid(axis='y')
    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='both', which='minor', labelsize=16)  
#     if not ylim_replacement:
#         _, ylim_replacement = ax.get_ylim()
#         ylim_replacement = ylim_replacement * 1.3
    #ax.set(ylim=[0, ylim_replacement])
    start, end = ax.get_ylim()
    stepsize = Stepsize_Cal(end) 
    ax.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax.set_xlabel('Expected Replacement Date (in Quarter)', fontsize=20)
    ax.set_ylabel('IPG Counts', fontsize=20)
    ax.legend(list('Sales in ' + df_reshape_low.columns[:len(dfos_qt)]) + list('Predicted Sales in ' + df_reshape_low.columns[-len(dfos_predict_qt_low)::]),
              loc='upper right', bbox_to_anchor=(1.45, 1), fontsize=20)
    
#     ax2 = ax.twinx()
#     (df_reshape_low/total).plot(kind='bar', stacked=True, ax=ax2, figsize=(20,12), fontsize=18, color=colors)
#     #df_reshape_low.plot.bar(stacked=True, ax=ax2, figsize=(20,12), fontsize=18)
#     plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
    
#     ax2.tick_params(axis='y', labelcolor='red', labelsize=20)
#     #ax2.set(ylim=[0, ylim_replacement/total])
#     start, end = ax2.get_ylim()
#     ax2.yaxis.set_ticks(np.arange(start, end, 0.01))
#     ax2.set_ylabel('Percentage of Total', fontsize=20, color='red')
    
#     ax.get_legend().remove()
#     ax2.legend(list('Sales in ' + df_reshape_low.columns[:len(dfos_qt)]) + list('Predicted Sales in ' + df_reshape_low.columns[-len(dfos_predict_qt_low)::]),
#                loc='upper right', bbox_to_anchor=(1.45, 1), fontsize=20)

    ax.set_title(r'$\bf{' + f"Replacement\ Timeline\ of\ {total}\ IPGs " + ProgramType + '}$' +
                 ' Based on Low Estimation of Predicted New Sales' + '\n-- ' + 
                 r'$\bf{' + estimate + '}$' +
                 ' expectation based on ' + impedance + ' = ' + imp_est, fontsize=20)
    plt.tick_params(labelright=True)
    #fig.savefig('figures/' + ProgramType + '_replacement_' + estimate + '_estimate_predict_low.png', bbox_inches='tight')
    
    ######### replacement count
    ax_rev.plot(df_reshape_low.index[df_reshape_low.index.isin(display_qt_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_qt_range)], **ls)
    ax_rev.tick_params(axis='both', which='major', labelsize=20)
    ax_rev.tick_params(axis='both', which='minor', labelsize=16)
    #ax_rev.set(ylim=[0, ylim_replacement])
    start, end = ax_rev.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_rev.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_rev.set_xlabel('Time of Sales (in Quarter)', fontsize=20)
    ax_rev.set_ylabel('IPG Counts', fontsize=20)
    
    ax_rev2.plot(df_reshape_low.index[df_reshape_low.index.isin(display_qt_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_qt_range)], **ls)
    ax_rev2.tick_params(axis='y', labelsize=20)
    #ax_rev2.set(ylim=[0, ylim_replacement]); 
    start, end = ax_rev2.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_rev2.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_rev2.set_ylabel('IPG Counts', fontsize=20)

    ######### total count
    if estimate == 'Earliest':
        dfos_qt_low = pd.concat([dfos_qt, dfos_predict_qt_low], axis=0, sort=True).sort_values(['Time in Quarter']).reset_index(drop=True)[['Time in Quarter', 'US Total']]
        dfos_qt_low = dfos_qt_low[dfos_qt_low['Time in Quarter'].isin(df_reshape_low.index[df_reshape_low.index.isin(display_qt_range)].tolist())]
        #ax_trev.plot(dfos_qt_low['Time in Quarter'], dfos_qt_low['US Total'], **dict(ls='dashed', linewidth=3, color='blue', marker='.', markersize=20))

    dfos_total_qt_low = pd.Series(dfos_qt_low['US Total'].tolist(), index=dfos_qt_low['Time in Quarter']).add(
        df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_qt_range)], fill_value=0)
    ax_trev.plot(dfos_total_qt_low.index, dfos_total_qt_low, **tls)
    ax_trev.plot(df_reshape_low.index[df_reshape_low.index.isin(display_qt_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_qt_range)], **ls)
    ax_trev.tick_params(axis='both', which='major', labelsize=20)
    ax_trev.tick_params(axis='both', which='minor', labelsize=16)
    #ax_trev.set(ylim=[0, ylim_replacement])
    start, end = ax_trev.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_trev.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_trev.set_xlabel('Time of Sales (in Quarter)', fontsize=20)
    ax_trev.set_ylabel('IPG Counts', fontsize=20)
    
    ax_trev2.plot(dfos_total_qt_low.index, dfos_total_qt_low, **tls)
    ax_trev2.plot(df_reshape_low.index[df_reshape_low.index.isin(display_qt_range)], df_reshape_low.sum(axis=1)[df_reshape_low.index.isin(display_qt_range)], **ls)
    ax_trev2.tick_params(axis='y', labelsize=20)
    #ax_trev2.set(ylim=[0, ylim_replacement]); 
    start, end = ax_trev2.get_ylim()
    stepsize = Stepsize_Cal(end)
    ax_trev2.yaxis.set_ticks(np.arange(start, end, stepsize))
    ax_trev2.set_ylabel('IPG Counts', fontsize=20)


ax_rev.grid(axis='y')
lgd = [(fitting + '\n' + estimate + ' expectation of replacement') 
       for fitting in ['Based on low estimation of predicted new sales']
       for estimate in ['& earliest', '& medium', '& latest']]
ax_rev.legend(lgd, loc='center', bbox_to_anchor=(1.4, 0.5), fontsize=16, ncol=1, handleheight=0)    
ax_rev.set_title(('US Orion IPG Replacement Count Prediction ' + ProgramType), fontsize=20, fontweight='bold')
# locs, labels = plt.xticks()
# ax_rev.set_xticklabels(labels, rotation=90)
for tick in ax_rev.get_xticklabels():
    tick.set_rotation(90)
#fig_rev.savefig('figures/' + ProgramType + '_replacement_count_predict_qt.png', bbox_inches='tight')

ax_trev.grid(axis='y')
lgd = [(rev + '\n' + fitting + '\n' + estimate + ' expectation of replacement)')
       for fitting in ['(based on low estimation of predicted new sales']
       for estimate in ['& earliest', '& medium', '& latest']
       for rev in ['New Sales + Replacement', 'Replacement Only']]
ax_trev.legend(lgd, loc='center', bbox_to_anchor=(1.4, 0.42), fontsize=16, ncol=1, handleheight=0)    
ax_trev.set_title(('US Orion Total Count Prediction (New Sales + Replacement) ' + ProgramType), fontsize=20, fontweight='bold')
# locs, labels = plt.xticks()
# ax_trev.set_xticklabels(labels, rotation=90)
for tick in ax_trev.get_xticklabels():
    tick.set_rotation(90)
#fig_trev.savefig('figures/' + ProgramType + '_total_count_predict_qt.png', bbox_inches='tight')

In [ ]:
#battery_replace_ratio = pd.Series(list(np.linspace(0.6, 0.85, num=5)) + [0.85]*(len(display_qt_range) - 5), index=display_qt_range)

df_replacement_qt = pd.DataFrame({'Earliest': ((df_reshape_qt_Earliest*battery_replace_ratio).add(
                                                df_reshape2_qt_Earliest*(1-battery_replace_ratio), fill_value=0)).round(),
                                  'Medium': ((df_reshape_qt_Medium*battery_replace_ratio).add(
                                              df_reshape2_qt_Medium*(1-battery_replace_ratio), fill_value=0)).round(),
                                  'Latest': ((df_reshape_qt_Latest*battery_replace_ratio).add(
                                              df_reshape2_qt_Latest*(1-battery_replace_ratio), fill_value=0)).round()})
df_replacement_qt.fillna(0, inplace=True)
df_replacement_qt.sort_index(inplace=True)

In [ ]:
#df_replacement_qt = df_replacement_qt.apply(lambda x: moving_average(x, 3), axis=0)

In [ ]:
replacement_predict_qt = df_replacement_qt.transpose().describe().transpose().sort_index()['mean']
replacement_predict_std_qt = df_replacement_qt.transpose().describe().transpose().sort_index()['std']

In [ ]:
replacement_predict_qt = pd.Series(0, index=display_qt_range).add(replacement_predict_qt, fill_value=0)
replacement_predict_std_qt = pd.Series(0, index=display_qt_range).add(replacement_predict_std_qt, fill_value=0)

In [ ]:
fig,ax = plt.subplots()
fig.set_size_inches(20,12)
ax2 = ax.twinx()
#plt.grid(axis='y')
ax.grid(axis='y')
# ax.set_axisbelow(True)
# ax.yaxis.grid(color='gray', linestyle='dashed')

display_skip =0 

ax.plot(replacement_predict_qt.index[replacement_predict_qt.index.isin(display_qt_range)], 
        replacement_predict_qt.values[replacement_predict_qt.index.isin(display_qt_range)],
        linewidth=3, marker='.', markersize=20, color='black')
ax.plot(df_replacement_qt['Earliest'].index[df_replacement_qt['Earliest'].index.isin(display_qt_range)],
        df_replacement_qt['Earliest'].values[df_replacement_qt['Earliest'].index.isin(display_qt_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='blue')
ax.plot(df_replacement_qt['Medium'].index[df_replacement_qt['Medium'].index.isin(display_qt_range)],
        df_replacement_qt['Medium'].values[df_replacement_qt['Medium'].index.isin(display_qt_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='green')
ax.plot(df_replacement_qt['Latest'].index[df_replacement_qt['Latest'].index.isin(display_qt_range)],
        df_replacement_qt['Latest'].values[df_replacement_qt['Latest'].index.isin(display_qt_range)], linestyle='dashed', linewidth=1, marker='.', markersize=10, color='purple')
ax.plot(dfos_r_qt['Time in Quarter'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)][display_skip:],
        dfos_r_qt['US Total'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)][display_skip:],
        linestyle='dotted', marker='.', markersize=10, color='red', alpha=0.6)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time (in Quarter)', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

ax2.set(ylim=[0, end]); 
ax2.tick_params(axis='y', labelsize=18)
ax2.yaxis.set_ticks(np.arange(0, end, stepsize))
ax2.set_ylabel('Orion IPG Counts', fontsize=20)

for tick in ax.get_xticklabels():
    tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life (average of a, b & c)',
                 '  a) Predicted Replacement (earliest, based on highest impedance)',
                 '  b) Predicted Replacement (medium, based on medium impedance)',
                 '  c) Predicted Replacement (latest, based on lowest impedance)',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)'],
                loc='best', fontsize=20)
#plt.savefig('figures/' + ProgramType + '_IPG_replacement_qt.png')

In [ ]:
fig,ax = plt.subplots()
fig.set_size_inches(20,12)
ax2 = ax.twinx()
#plt.grid(axis='y')
ax.grid(axis='y')
# ax.set_axisbelow(True)
# ax.yaxis.grid(color='gray', linestyle='dashed')

display_skip = 0 

ax.plot(replacement_predict_qt.index[replacement_predict_qt.index.isin(display_qt_range)], 
        replacement_predict_qt.values[replacement_predict_qt.index.isin(display_qt_range)],
        linewidth=3, marker='.', markersize=20, color='black')
ax.plot(dfos_r_qt['Time in Quarter'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)][display_skip:],
        dfos_r_qt['US Total'][dfos_r_qt['Time in Quarter'].isin(display_qt_range)][display_skip:],
        linestyle='dotted', marker='.', markersize=10, color='red', alpha=0.6)
ax.errorbar(replacement_predict_qt.index[replacement_predict_qt.index.isin(display_qt_range)], 
            replacement_predict_qt.values[replacement_predict_qt.index.isin(display_qt_range)],
            yerr = replacement_predict_std_qt.values[replacement_predict_std_qt.index.isin(display_qt_range)],
            linewidth=2, color='gray', barsabove=True, capsize=5)

ax.tick_params(axis='both', which='major', labelsize=18)
ax.tick_params(axis='both', which='minor', labelsize=12)
start, end = ax.get_ylim()
end = end * 1.4
stepsize = Stepsize_Cal(end)
ax.set(ylim=[0, end]); 
ax.yaxis.set_ticks(np.arange(0, end, stepsize))
ax.set_xlabel('Time (in Quarter)', fontsize=20)
ax.set_ylabel('Orion IPG Counts', fontsize=20)

ax2.set(ylim=[0, end]); 
ax2.tick_params(axis='y', labelsize=18)
ax2.yaxis.set_ticks(np.arange(0, end, stepsize))
ax2.set_ylabel('Orion IPG Counts', fontsize=20)

for tick in ax.get_xticklabels():
    tick.set_rotation(90)
ax.set_title(('US Orion IPG Replacement Prediction Based on IPG Longevity ' + ProgramType), fontsize=20, fontweight='bold')
lgd = ax.legend(['Number of IPGs that Need to Be Replaced due to Battery End of Life',
                 'Actual Total Replacements\n(including those due to battery end of life and all other reasons)',
                 'Range of Predicted Replacement'],
                loc='best', fontsize=20)
#plt.savefig('figures/' + ProgramType + '_IPG_replacement_qt_std.png')